# NDT7 EDA — Philippines Broadband & Mobile/Cellular (Combined)

Full exploratory analysis of Philippines NDT7 (M-Lab) data, mirroring the structure of
`notebooks/ookla/philippines_eda.ipynb` and the other tigger countries' EDA notebooks
(Cambodia/Thailand/Vietnam) so results are directly comparable section-by-section, mirroring
the prep/EDA split established by `philippines_ndt7_prep.ipynb`.

**Not yet executed in this repo** — Philippines raw data (`data/ph/mlab_ph_clean.parquet`) is
not available locally. Needs a run+verify pass before use, e.g. by whoever has the full local
dataset.

**Input:** `data/exports/ndt7_philippines_province_quarterly.csv` (Broadband) and
`data/exports/ndt7_mobile_philippines_province_quarterly.csv` (Mobile) — both produced by
`philippines_ndt7_prep.ipynb`. This notebook does no raw-data processing for the standard
template sections below — it picks up where the prep notebook leaves off. The PH-specific
sections appended at the end of this notebook (after both Broadband and Mobile/Cellular parts)
DO query the raw parquet directly, same as the original `ndt7_ph_main.ipynb` /
`ndt7_ph_manila.ipynb` / `ndt7_ph_paper.ipynb` they were ported from — they need ISP/hour/true
province-level detail that isn't in the aggregated CSV.

**Two structural differences from Cambodia/Thailand/Vietnam, both deliberate — see
`philippines_ndt7_prep.ipynb` intro for the full rationale:**

1. **`is_reliable` = `total_tests >= 100` only** — no tile-binning was ever built for PH, so
   there's no `n_tiles` and no `n_tiles>=5` bar here. Not directly comparable to the other
   tigger countries' reliability metric.
2. **`province` in the standard sections below = the 17 PH administrative regions**, not the 80
   ADM1 provinces — this is the only granularity with GDP/density/tier reference data
   (`data/reference/philippines_reference.csv`, matching what
   `notebooks/ookla/philippines_eda.ipynb` already does). True 80-province and city-level detail
   is NOT lost — it's covered in depth by the PH-specific sections at the end of this notebook
   (Manila vs Rest of PH, top/bottom provinces, island-group divide), which is why those were
   kept as additions rather than dropped to force a 1:1 template match with the other tigger
   countries.

**Reliability result: not yet known** — fill in after the first execution (this replaces
Cambodia's "0% reliability" finding, which does NOT apply here; PH's `is_reliable` formula and
underlying data volume are both different).


## ISP Overview — Philippines

Raw NDT7 ISP names from `ph/mlab_ph_clean.parquet` (pre-classification, as delivered
by the ISP-matching step) — a diagnostic view of who's actually in the data before it's rolled
up to province-quarter, split by `network_type` and ranked by test volume.

In [ ]:
import pyarrow.parquet as pq

RAW_ISP_PARQUET = '/home/chissanupun/Desktop/code/lab/cnc/data-science/internet-measurement/data/ph/mlab_ph_clean.parquet'
isp_df = pq.read_table(RAW_ISP_PARQUET, columns=['isp', 'network_type']).to_pandas()
isp_counts = (
    isp_df.groupby(['network_type', 'isp']).size().rename('n_tests')
    .reset_index().sort_values(['network_type', 'n_tests'], ascending=[True, False])
)
print(f"{isp_df['isp'].nunique()} unique ISP names, {len(isp_df):,} raw test rows")
isp_counts

## Province Reference Data

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import os

PROVINCE_REF_PATH = '../../../data/reference/philippines_reference.csv'

prov_ref = pd.read_csv(PROVINCE_REF_PATH)
prov_ref['gdp_per_capita_usd_2021'] = prov_ref['gdp_per_capita_thb_2021']  # already USD-equivalent in source data

print(f"Provinces loaded: {len(prov_ref)}")
print(f"Regions: {sorted(prov_ref['region'].unique())}")
print()
print(prov_ref[['province_en','region','pop_2024','density_per_km2','gdp_per_capita_thb_2021','internet_tier']].to_string(index=False))

### Population and GDP by Region — Summary

In [ ]:
region_summary = (
    prov_ref.groupby('region')
    .agg(
        provinces=('province_en', 'count'),
        total_pop=('pop_2024', 'sum'),
        avg_density=('density_per_km2', 'mean'),
        avg_gdp_per_cap_thb=('gdp_per_capita_thb_2021', 'mean'),
        avg_gdp_per_cap_usd=('gdp_per_capita_usd_2021', 'mean'),
    )
    .sort_values('avg_gdp_per_cap_thb', ascending=False)
    .round(0)
)

region_summary['total_pop_M'] = (region_summary['total_pop'] / 1e6).round(2)
print(region_summary[['provinces','total_pop_M','avg_density','avg_gdp_per_cap_thb','avg_gdp_per_cap_usd']].to_string())

### GDP per Capita vs Population Density — by Region

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

region_colors = {
    'Luzon': '#1f77b4',
    'Visayas': '#ff7f0e',
    'Mindanao': '#2ca02c',
}

for _, row in prov_ref.iterrows():
    c = region_colors.get(row['region'], 'gray')
    axes[0].scatter(row['density_per_km2'], row['gdp_per_capita_thb_2021'] / 1000,
                    color=c, alpha=0.7, s=row['pop_2024'] / 50000, edgecolors='white', linewidth=0.5)
    if row['gdp_per_capita_thb_2021'] > prov_ref['gdp_per_capita_thb_2021'].quantile(0.85):
        axes[0].annotate(row['province_en'], (row['density_per_km2'], row['gdp_per_capita_thb_2021'] / 1000),
                         fontsize=6.5, ha='left', va='bottom', alpha=0.9)

axes[0].set_xlabel('Population Density (per km²)')
axes[0].set_ylabel('GDP per Capita (000s USD, 2023)')
axes[0].set_title('GDP per Capita vs Density\n(bubble size = population)')
axes[0].set_xscale('log')

r_gdp = prov_ref.groupby('region')['gdp_per_capita_thb_2021'].mean().sort_values(ascending=True) / 1000
colors = [region_colors.get(r, 'gray') for r in r_gdp.index]
r_gdp.plot(kind='barh', ax=axes[1], color=colors)
axes[1].set_xlabel('Avg GDP per Capita (000s USD, 2023)')
axes[1].set_title('Avg GDP per Capita by Region')
axes[1].axvline(prov_ref['gdp_per_capita_thb_2021'].mean() / 1000, color='black', linestyle='--', linewidth=1, label='National avg')
axes[1].legend()

patches = [mpatches.Patch(color=v, label=k) for k, v in region_colors.items()]
axes[0].legend(handles=patches, fontsize=7, loc='upper left')

plt.tight_layout()
os.makedirs('../../../outputs/ndt7/philippines', exist_ok=True)
plt.savefig('../../../outputs/ndt7/philippines/09_gdp_vs_density.png', dpi=150, bbox_inches='tight')
plt.show()

### Internet Tier Distribution — Assumptions Map

In [ ]:
CAMBODIA_GEOJSON = '../../../data/geo/philippines_provinces.geojson'
thailand_gdf = gpd.read_file(CAMBODIA_GEOJSON).to_crs(4326)

tier_map = prov_ref[['province_en','internet_tier','gdp_per_capita_thb_2021','region']].copy()
geo_with_tier = thailand_gdf.merge(
    tier_map.rename(columns={'province_en': 'name'}),
    on='name', how='left'
)

missing = geo_with_tier[geo_with_tier['internet_tier'].isna()]['name'].tolist()
if missing:
    print("Unmatched provinces (need manual fix):", missing)

tier_colors = {1: '#1d3557', 2: '#457b9d', 3: '#a8dadc', 4: '#e63946'}
geo_with_tier['tier_color'] = geo_with_tier['internet_tier'].map(tier_colors)

fig, ax = plt.subplots(1, figsize=(10, 12))
geo_with_tier.plot(color=geo_with_tier['tier_color'].fillna('lightgray'), ax=ax, edgecolor='white', linewidth=0.5)

patches = [
    mpatches.Patch(color='#1d3557', label='Tier 1 — Highest GDP per capita'),
    mpatches.Patch(color='#457b9d', label='Tier 2 — Above average'),
    mpatches.Patch(color='#a8dadc', label='Tier 3 — Average'),
    mpatches.Patch(color='#e63946', label='Tier 4 — Below average'),
]
ax.legend(handles=patches, loc='lower right', fontsize=9)
ax.set_title('Expected Fixed Broadband Performance Tier by Province\n(based on GDP per capita, density, urbanization)', fontsize=12)
ax.set_axis_off()
plt.tight_layout()

plt.savefig('../../../outputs/ndt7/philippines/10_tier_speed_expected.png', dpi=150, bbox_inches='tight')
plt.show()

### Top & Bottom Provinces by GDP per Capita — with Internet Tier

In [ ]:
top10 = prov_ref.nlargest(10, 'gdp_per_capita_thb_2021')[['province_en','region','pop_2024','gdp_per_capita_thb_2021','internet_tier']]
bot10 = prov_ref.nsmallest(10, 'gdp_per_capita_thb_2021')[['province_en','region','pop_2024','gdp_per_capita_thb_2021','internet_tier']]

print("=== TOP 10 by GDP per Capita ===")
print(top10.to_string(index=False))
print()
print("=== BOTTOM 10 by GDP per Capita ===")
print(bot10.to_string(index=False))

---
# Part 1 — Broadband

## Full EDA — NDT7 Broadband, Philippines (2023 Q1 – 2025 Q4)

### 1. Load Province x Quarter Data

In [ ]:
import warnings
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from scipy import stats
warnings.filterwarnings('ignore')

NDT7_CSV = '../../../data/exports/ndt7_philippines_province_quarterly.csv'
GEOJSON  = '../../../data/geo/philippines_provinces.geojson'
PROV_REF = '../../../data/reference/philippines_reference.csv'

Q_TO_MONTH = {1: 1, 2: 4, 3: 7, 4: 10}

thailand_gdf = gpd.read_file(GEOJSON).to_crs(4326)

master = pd.read_csv(NDT7_CSV)
master = master.rename(columns={'province': 'name', 'quarter': 'label', 'quarter.1': 'quarter'})
master['month'] = master['quarter'].map(Q_TO_MONTH)
master['period_start'] = pd.to_datetime(dict(year=master['year'], month=master['month'], day=1))

prov_ref = pd.read_csv(PROV_REF)
prov_ref['gdp_per_capita_usd'] = prov_ref['gdp_per_capita_thb_2021']

print(f"Master shape: {master.shape}")
print(f"Date range: {master['period_start'].min().date()} -> {master['period_start'].max().date()}")
print(f"Quarters:   {sorted(master['label'].unique())}")
print(f"Provinces:  {master['name'].nunique()} (of {len(prov_ref)} in reference — NDT7 has zero test volume for the rest)")

In [ ]:
# is_reliable already computed upstream (philippines_ndt7_prep.ipynb) —
# NOTE: unlike KH/TH/VN, Philippines' is_reliable is `total_tests >= 100` ONLY (no tile-binning,
# no n_tiles — see prep notebook intro). Not directly comparable to the other tigger countries'
# `n_tiles>=5` bar; treat PH's reliability rate as its own methodology.
reliable_count = master.groupby('label')['is_reliable'].sum()
print('Reliable region-quarters per label:')
print(reliable_count.to_string())
print(f"\nOverall reliable: {master['is_reliable'].sum()} / {len(master)} ({master['is_reliable'].mean():.1%})")


### 2. Data Quality — Coverage per Province

In [ ]:
expected_n = len(prov_ref)
expected = expected_n * master['label'].nunique()
print(f"Expected rows (if every province had every quarter): {expected} | Actual: {len(master)} | Missing: {expected - len(master)}")

all_provinces = set(thailand_gdf['name'])
provinces_with_data = set(master['name'].unique())
zero_coverage = sorted(all_provinces - provinces_with_data)
if zero_coverage:
    print(f"\nProvinces with ZERO NDT7 test volume in any quarter ({len(zero_coverage)}): {zero_coverage}")
else:
    print("\nEvery province has at least some NDT7 data in at least one quarter.")

low_test = master[master['total_tests'] < 100][['name','label','total_tests','n_tiles']].sort_values('total_tests')
print(f"\nLow-test quarters (< 100 tests) — {len(low_test)} of {len(master)} province-quarter rows")
print(low_test.head(15).to_string(index=False))

In [ ]:
# Drop unreliable province-quarters before any stats/plots use master downstream — same rule as every
# other country/source. For Philippines this empties master entirely (0 reliable rows); sections below
# report that explicitly rather than silently rendering blank charts.
master = master[master['is_reliable']].copy()
print(f"Filtered to reliable province-quarters: {len(master)} rows kept")
HAS_RELIABLE_DATA = len(master) > 0
if not HAS_RELIABLE_DATA:
    print("\nNo province-quarters clear the reliability threshold — sections 3-14 below will report "
          "'no reliable data' instead of plotting.")

### 3. Speed Distribution — All Provinces, All Quarters

In [ ]:
if not HAS_RELIABLE_DATA:
    print("No reliable data for Broadband — skipping speed-distribution plot.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].hist(master['avg_d_mbps'].dropna(), bins=60, color='#457b9d', edgecolor='none', alpha=0.85)
    axes[0].axvline(master['avg_d_mbps'].median(), color='red', ls='--', lw=1.5, label=f'Median {master["avg_d_mbps"].median():.0f} Mbps')
    axes[0].set_xlabel('Avg Download (Mbps)'); axes[0].set_title('Download Speed Distribution'); axes[0].legend()

    axes[1].hist(master['avg_u_mbps'].dropna(), bins=60, color='#2a9d8f', edgecolor='none', alpha=0.85)
    axes[1].axvline(master['avg_u_mbps'].median(), color='red', ls='--', lw=1.5, label=f'Median {master["avg_u_mbps"].median():.0f} Mbps')
    axes[1].set_xlabel('Avg Upload (Mbps)'); axes[1].set_title('Upload Speed Distribution'); axes[1].legend()

    axes[2].hist(master['avg_lat_ms_wt'].dropna(), bins=60, color='#e76f51', edgecolor='none', alpha=0.85)
    axes[2].axvline(master['avg_lat_ms_wt'].median(), color='navy', ls='--', lw=1.5, label=f'Median {master["avg_lat_ms_wt"].median():.0f} ms')
    axes[2].set_xlabel('Avg Latency (ms)'); axes[2].set_title('Latency Distribution'); axes[2].legend()

    plt.suptitle('Broadband Metric Distributions — All Provinces, All Quarters (2023–2025)', fontsize=13)
    plt.tight_layout()

    plt.savefig('../../../outputs/ndt7/philippines/11_upload_speed_dist.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(master[['avg_d_mbps','avg_u_mbps','avg_lat_ms_wt','total_tests']].describe().round(2))

### 4. Province Ranking — Mean Download Speed (All Quarters)

In [ ]:
region_colors = {
    'Luzon': '#1f77b4',
    'Visayas': '#ff7f0e',
    'Mindanao': '#2ca02c',
}

if not HAS_RELIABLE_DATA:
    prov_mean = pd.DataFrame(columns=['name','region','mean_dl','mean_ul','mean_lat','mean_tests'])
    print("No reliable data for Broadband — skipping province-ranking plot.")
else:
    prov_mean = (
        master.groupby(['name','region'])
        .agg(
            mean_dl=('avg_d_mbps','mean'),
            mean_ul=('avg_u_mbps','mean'),
            mean_lat=('avg_lat_ms_wt','mean'),
            mean_tests=('total_tests','mean'),
        )
        .reset_index()
        .sort_values('mean_dl', ascending=True)
    )

    fig, ax = plt.subplots(figsize=(14, 22))
    colors = [region_colors.get(r,'gray') for r in prov_mean['region']]
    bars = ax.barh(prov_mean['name'], prov_mean['mean_dl'], color=colors, edgecolor='none')
    for bar, val in zip(bars, prov_mean['mean_dl']):
        ax.text(bar.get_width() + 1.5, bar.get_y() + bar.get_height()/2, f'{val:.0f}', va='center', fontsize=7)
    ax.axvline(prov_mean['mean_dl'].mean(), color='black', ls='--', lw=1, label=f'National mean {prov_mean["mean_dl"].mean():.0f} Mbps')
    patches = [mpatches.Patch(color=v, label=k) for k, v in region_colors.items()]
    ax.legend(handles=patches, loc='lower right', fontsize=9)
    ax.set_xlabel('Mean Download Speed (Mbps)')
    ax.set_title('Broadband Download Speed by Province\n(weighted avg per quarter, averaged 2023 Q1 – 2025 Q4)', fontsize=12)
    plt.tight_layout()
    plt.savefig('../../../outputs/ndt7/philippines/12_province_ranking_mean.png', dpi=150, bbox_inches='tight')
    plt.show()

### 5. Time Series — Download Speed Heatmap (Province × Quarter)

In [ ]:
if not HAS_RELIABLE_DATA:
    print("No reliable data for Broadband — skipping heatmap.")
else:
    pivot_dl = master.pivot_table(index='name', columns='period_start', values='avg_d_mbps')
    pivot_dl.columns = [pd.Timestamp(c).strftime('%Y-Q') + str((pd.Timestamp(c).month-1)//3+1) for c in pivot_dl.columns]
    pivot_dl = pivot_dl.loc[prov_mean.sort_values('mean_dl', ascending=False)['name']]

    fig, ax = plt.subplots(figsize=(16, 22))
    im = ax.imshow(pivot_dl.values, aspect='auto', cmap='YlOrRd', interpolation='nearest')
    plt.colorbar(im, ax=ax, label='Avg Download (Mbps)', shrink=0.5)
    ax.set_xticks(range(len(pivot_dl.columns)))
    ax.set_xticklabels(pivot_dl.columns, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(len(pivot_dl.index)))
    ax.set_yticklabels(pivot_dl.index, fontsize=8)
    ax.set_title('Broadband Download Speed Heatmap\n(province × quarter, sorted by mean speed)', fontsize=12)
    plt.tight_layout()
    plt.savefig('../../../outputs/ndt7/philippines/13_heatmap_quarterly.png', dpi=150, bbox_inches='tight')
    plt.show()

### 6. Regional Comparison — Download, Upload, Latency

In [ ]:
region_order = ['Luzon', 'Visayas', 'Mindanao']

if not HAS_RELIABLE_DATA:
    print("No reliable data for Broadband — skipping regional boxplot.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    for ax, metric, label, color_key in [
        (axes[0], 'avg_d_mbps',   'Download (Mbps)', '#457b9d'),
        (axes[1], 'avg_u_mbps',   'Upload (Mbps)',   '#2a9d8f'),
        (axes[2], 'avg_lat_ms_wt','Latency (ms)',    '#e76f51'),
    ]:
        data = [master[master['region']==r][metric].dropna().values for r in region_order]
        bp = ax.boxplot(data, patch_artist=True, notch=False, medianprops=dict(color='white', linewidth=2))
        for patch, region in zip(bp['boxes'], region_order):
            patch.set_facecolor(region_colors[region])
            patch.set_alpha(0.85)
        ax.set_xticklabels(region_order, rotation=30, ha='right', fontsize=8)
        ax.set_ylabel(label)
        ax.set_title(f'{label} by Region')
    plt.suptitle('Broadband Performance by Region — All Quarters (2023–2025)', fontsize=13)
    plt.tight_layout()
    plt.savefig('../../../outputs/ndt7/philippines/14_regional_boxplot.png', dpi=150, bbox_inches='tight')
    plt.show()

### 7. Download Speed Trend — Top, Bottom & Interesting Provinces

In [ ]:
if not HAS_RELIABLE_DATA:
    print("No reliable data for Broadband — skipping time-series trend plot.")
else:
    latest_q = master['label'].max()
    latest = master[master['label'] == latest_q]
    top5 = latest.nlargest(5, 'avg_d_mbps')['name'].tolist()
    bot5 = latest.nsmallest(5, 'avg_d_mbps')['name'].tolist()

    fig, axes = plt.subplots(2, 1, figsize=(16, 12), sharex=True)
    groups = [
        (f'Top 5 Fastest ({latest_q})', top5, '#e63946'),
        (f'Bottom 5 Slowest ({latest_q})', bot5, '#6d6875'),
    ]
    for ax, (title, provinces, base_color) in zip(axes, groups):
        for prov in provinces:
            df_p = master[master['name']==prov].sort_values('label')
            if df_p.empty: continue
            ax.plot(df_p['period_start'], df_p['avg_d_mbps'], marker='o', markersize=5, label=prov, linewidth=1.8)
        ax.set_ylabel('Avg Download (Mbps)')
        ax.set_title(title)
        ax.legend(fontsize=8, loc='upper left')
        ax.grid(axis='y', alpha=0.3)
        ax.tick_params(axis='x', rotation=30)
    plt.suptitle('Download Speed Trend (2023 Q1 – 2025 Q4)', fontsize=13)
    plt.tight_layout()
    plt.savefig('../../../outputs/ndt7/philippines/15_time_series.png', dpi=150, bbox_inches='tight')
    plt.show()

### 8. GDP per Capita vs Download Speed — Does Money = Speed?

In [ ]:
if not HAS_RELIABLE_DATA:
    corr_df = pd.DataFrame(columns=['name','region','mean_dl','gdp_per_capita_thb_2021','internet_tier','pop_2024','density_per_km2'])
    r, p, r2, p2 = float('nan'), float('nan'), float('nan'), float('nan')
    print("No reliable data for Broadband — skipping GDP/density correlation plot.")
else:
    corr_df = prov_mean.merge(
        prov_ref[['province_en','gdp_per_capita_thb_2021','internet_tier','pop_2024','density_per_km2']].rename(columns={'province_en':'name'}),
        on='name', how='left'
    )

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    for _, row in corr_df.iterrows():
        c = region_colors.get(row['region'], 'gray')
        axes[0].scatter(row['gdp_per_capita_thb_2021']/1000, row['mean_dl'],
                        color=c, s=row['pop_2024']/80000, alpha=0.7, edgecolors='white', lw=0.5)
        if row['gdp_per_capita_thb_2021'] > corr_df['gdp_per_capita_thb_2021'].quantile(0.9) or row['mean_dl'] > corr_df['mean_dl'].quantile(0.9) or row['mean_dl'] < corr_df['mean_dl'].quantile(0.1):
            axes[0].annotate(row['name'], (row['gdp_per_capita_thb_2021']/1000, row['mean_dl']), fontsize=6.5, ha='left', va='bottom')

    x = corr_df['gdp_per_capita_thb_2021'].dropna()/1000
    y = corr_df.loc[x.index,'mean_dl']
    if x.nunique() > 1:
        slope, intercept, r, p, _ = stats.linregress(x, y)
        xline = np.linspace(x.min(), x.max(), 100)
        axes[0].plot(xline, slope*xline+intercept, 'k--', lw=1.5, label=f'OLS (r={r:.2f}, p={p:.3f})')
        ols_handle = [plt.Line2D([0],[0],color='k',ls='--',label=f'OLS r={r:.2f}')]
    else:
        r, p = float('nan'), float('nan')
        axes[0].text(0.5, 0.05, 'GDP per capita uniform across regions — no regression', transform=axes[0].transAxes, ha='center', fontsize=8, style='italic')
        ols_handle = []
    axes[0].set_xlabel('GDP per Capita (000s USD, 2023)')
    axes[0].set_ylabel('Mean Download Speed (Mbps)')
    axes[0].set_title('GDP per Capita vs Avg Download Speed')
    patches = [mpatches.Patch(color=v, label=k) for k,v in region_colors.items()]
    axes[0].legend(handles=patches+ols_handle, fontsize=7)

    for _, row in corr_df.iterrows():
        c = region_colors.get(row['region'],'gray')
        axes[1].scatter(row['density_per_km2'], row['mean_dl'], color=c, s=row['pop_2024']/80000, alpha=0.7, edgecolors='white', lw=0.5)
        if row['density_per_km2'] > corr_df['density_per_km2'].quantile(0.9) or row['mean_dl'] > corr_df['mean_dl'].quantile(0.9) or row['mean_dl'] < corr_df['mean_dl'].quantile(0.1):
            axes[1].annotate(row['name'], (row['density_per_km2'], row['mean_dl']), fontsize=6.5, ha='left', va='bottom')

    x2 = corr_df['density_per_km2'].dropna()
    y2 = corr_df.loc[x2.index,'mean_dl']
    if x2.nunique() > 1:
        s2,i2,r2,p2,_ = stats.linregress(np.log1p(x2), y2)
        xline2 = np.linspace(x2.min(), x2.max(), 200)
        axes[1].plot(xline2, s2*np.log1p(xline2)+i2, 'k--', lw=1.5, label=f'OLS log (r={r2:.2f}, p={p2:.3f})')
    else:
        r2, p2 = float('nan'), float('nan')
    axes[1].set_xlabel('Population Density (per km²)')
    axes[1].set_xscale('log')
    axes[1].set_ylabel('Mean Download Speed (Mbps)')
    axes[1].set_title('Population Density vs Avg Download Speed (log scale)')
    axes[1].legend(fontsize=7)

    plt.suptitle('Economic & Demographic Correlates of Broadband Performance', fontsize=13)
    plt.tight_layout()
    plt.savefig('../../../outputs/ndt7/philippines/16_gdp_speed_scatter.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Pearson r (GDP vs download):     {r:.3f}  p={p:.4f}")
    print(f"Pearson r (log density vs download): {r2:.3f}  p={p2:.4f}")

### 8.5 Multivariate Model — GDP + Density + Tier + Region Together

In [ ]:
import statsmodels.formula.api as smf

if not HAS_RELIABLE_DATA or len(corr_df.dropna(subset=['mean_dl'])) < 5:
    print("No reliable data for Broadband — skipping multivariate model (need >=5 observations to fit).")
else:
    model_df = corr_df.dropna(subset=['mean_dl', 'gdp_per_capita_thb_2021', 'density_per_km2', 'region', 'internet_tier']).copy()
    model_df['log_gdp'] = np.log(model_df['gdp_per_capita_thb_2021'])
    model_df['log_density'] = np.log1p(model_df['density_per_km2'])

    terms = []
    if model_df['log_gdp'].nunique() > 1:
        terms.append('log_gdp')
    if model_df['log_density'].nunique() > 1:
        terms.append('log_density')
    if model_df['internet_tier'].nunique() > 1:
        terms.append("C(internet_tier)")
    if model_df['region'].nunique() > 1:
        terms.append('C(region)')

    if not terms:
        print("No varying predictors — skipping multivariate model.")
    else:
        formula = 'mean_dl ~ ' + ' + '.join(terms)
        mv_model = smf.ols(formula, data=model_df).fit()
        print(mv_model.summary())
        print(f"\nMultivariate R\u00b2 = {mv_model.rsquared:.3f}  (bivariate GDP-only r\u00b2 = {r**2:.3f})")
        print(f"Adjusted R\u00b2 = {mv_model.rsquared_adj:.3f}")

### 9. Upload/Download Ratio — Symmetry Check

In [ ]:
if not HAS_RELIABLE_DATA:
    print("No reliable data for Broadband — skipping UL/DL ratio plot.")
else:
    master['ul_dl_ratio'] = master['avg_u_mbps'] / master['avg_d_mbps']
    ratio_mean = master.groupby('name')['ul_dl_ratio'].mean().sort_values(ascending=False).reset_index()
    ratio_mean = ratio_mean.merge(prov_ref[['province_en','region']].rename(columns={'province_en':'name'}), on='name', how='left')

    fig, ax = plt.subplots(figsize=(14, 20))
    colors = [region_colors.get(r,'gray') for r in ratio_mean['region']]
    ax.barh(ratio_mean['name'], ratio_mean['ul_dl_ratio'], color=colors, alpha=0.85)
    ax.axvline(1.0, color='red', ls='--', lw=1.5, label='Symmetric (1:1)')
    ax.axvline(ratio_mean['ul_dl_ratio'].mean(), color='black', ls=':', lw=1, label=f'Mean ratio {ratio_mean["ul_dl_ratio"].mean():.2f}')
    ax.set_xlabel('Upload / Download Ratio')
    ax.set_title('Upload/Download Symmetry Ratio by Province\n(ratio > 0.8 = unusually symmetric; > 1.0 = upload > download)')
    patches = [mpatches.Patch(color=v, label=k) for k,v in region_colors.items()]
    ax.legend(handles=patches+[
        plt.Line2D([0],[0],color='red',ls='--',label='Symmetric 1:1'),
        plt.Line2D([0],[0],color='black',ls=':',label=f'Mean {ratio_mean["ul_dl_ratio"].mean():.2f}')
    ], fontsize=8, loc='lower right')
    plt.tight_layout()
    plt.savefig('../../../outputs/ndt7/philippines/17_uldl_ratio_bar.png', dpi=150, bbox_inches='tight')
    plt.show()

    divergent_ratio = ratio_mean[ratio_mean['ul_dl_ratio'] > 0.8]
    print(f"Provinces with UL/DL ratio > 0.8 ({len(divergent_ratio)}):")
    print(divergent_ratio[['name','region','ul_dl_ratio']].to_string(index=False))

### 10. Province Summary — Tier & UL/DL Ratio

In [ ]:
from scipy import stats
import numpy as np

if not HAS_RELIABLE_DATA:
    high_ratio = pd.Series(dtype=float)
    prov_mean = pd.DataFrame(columns=['name','region','mean_dl','mean_ul','mean_lat','mean_tests','mean_ratio','tier'])
    print("No reliable data for Broadband — skipping province summary.")
else:
    master['ul_dl_ratio'] = master['avg_u_mbps'] / master['avg_d_mbps']
    high_ratio = master.groupby('name')['ul_dl_ratio'].mean()

    prov_mean = (
        master.groupby(['name','region'])
        .agg(
            mean_dl=('avg_d_mbps','mean'),
            mean_ul=('avg_u_mbps','mean'),
            mean_lat=('avg_lat_ms_wt','mean'),
            mean_tests=('total_tests','mean'),
            mean_ratio=('ul_dl_ratio','mean'),
        )
        .reset_index()
        .sort_values('mean_dl', ascending=False)
    )
    prov_mean['tier'] = prov_mean['name'].map(prov_ref.set_index('province_en')['internet_tier'].to_dict())

### 12. Full Province Summary Table

In [ ]:
if not HAS_RELIABLE_DATA:
    print("No reliable data for Broadband — nothing to summarize.")
else:
    summary = prov_mean.merge(
        prov_ref[['province_en','pop_2024','gdp_per_capita_thb_2021','internet_tier']].rename(columns={'province_en':'name'}),
        on='name', how='left'
    ).copy()
    summary['ul_dl_ratio'] = summary['name'].map(high_ratio).round(3)
    summary = summary.sort_values('mean_dl', ascending=False).round(2)
    print(summary[[
        'name','region','internet_tier','mean_dl','mean_ul','mean_lat','ul_dl_ratio','mean_tests','gdp_per_capita_thb_2021'
    ]].to_string(index=False))

### 13. UL/DL Symmetry — Fiber Penetration Evidence

In [ ]:
if not HAS_RELIABLE_DATA:
    print("No reliable data for Broadband — skipping UL/DL symmetry deep-dive.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    ratio_sorted = prov_mean.sort_values('mean_ratio', ascending=True)
    colors = [region_colors.get(r,'gray') for r in ratio_sorted['region']]
    axes[0].barh(ratio_sorted['name'], ratio_sorted['mean_ratio'], color=colors, alpha=0.85)
    axes[0].axvline(prov_mean['mean_ratio'].mean(), color='black', ls='--', lw=1, label=f"National mean {prov_mean['mean_ratio'].mean():.2f}")
    axes[0].axvline(1.0, color='red', ls=':', lw=1, label='Symmetric 1:1')
    axes[0].set_xlabel('UL / DL Ratio')
    axes[0].set_title('UL/DL Symmetry Ratio by Province')
    patches = [mpatches.Patch(color=v,label=k) for k,v in region_colors.items()]
    axes[0].legend(handles=patches + [
        plt.Line2D([0],[0],color='black',ls='--',label=f"Mean {prov_mean['mean_ratio'].mean():.2f}"),
        plt.Line2D([0],[0],color='red',ls=':',label='Symmetric 1:1'),
    ], fontsize=7, loc='lower right')

    axes[1].scatter(prov_ref.set_index('province_en').loc[prov_mean['name'],'density_per_km2'].values,
                    prov_mean['mean_ratio'].values,
                    c=[region_colors.get(r,'gray') for r in prov_mean['region']],
                    s=80, alpha=0.7, edgecolors='white', lw=0.5)
    for _, row in prov_mean.iterrows():
        if row['mean_ratio'] > prov_mean['mean_ratio'].quantile(0.9) or row['mean_ratio'] < prov_mean['mean_ratio'].quantile(0.1):
            density = prov_ref.loc[prov_ref['province_en']==row['name'],'density_per_km2'].values
            if len(density):
                axes[1].annotate(row['name'], (density[0], row['mean_ratio']), fontsize=6.5, ha='left', va='bottom')
    axes[1].set_xscale('log')
    axes[1].set_xlabel('Population Density (per km², log scale)')
    axes[1].set_ylabel('UL/DL Ratio')
    axes[1].set_title('Density vs UL/DL Symmetry')
    axes[1].legend(handles=patches, fontsize=7)

    plt.suptitle('Upload/Download Symmetry — Broadband', fontsize=13)
    plt.tight_layout()
    plt.savefig('../../../outputs/ndt7/philippines/19_uldl_symmetry_ratio.png', dpi=150, bbox_inches='tight')
    plt.show()

### 14. Temporal: Speed Growth 2023 Q1 → 2025 Q4

In [ ]:
if not HAS_RELIABLE_DATA:
    print("No reliable data for Broadband — skipping temporal growth plot.")
else:
    first = master[master['label']=='2023-Q1'].set_index('name')['avg_d_mbps']
    last  = master[master['label']=='2025-Q4'].set_index('name')['avg_d_mbps']
    growth = ((last - first) / first * 100).dropna().sort_values(ascending=False)

    if growth.empty:
        print("No province has both a 2023-Q1 and 2025-Q4 reliable observation — skipping growth plot.")
    else:
        growth_df = growth.reset_index()
        growth_df.columns = ['name','pct_growth']
        growth_df = growth_df.merge(prov_ref[['province_en','region','internet_tier']].rename(columns={'province_en':'name'}), on='name', how='left')

        fig, ax = plt.subplots(figsize=(14, 20))
        colors = [region_colors.get(r,'gray') for r in growth_df['region']]
        bars = ax.barh(growth_df['name'], growth_df['pct_growth'], color=colors, alpha=0.85)
        ax.axvline(0, color='black', lw=0.8)
        ax.axvline(growth_df['pct_growth'].mean(), color='red', ls='--', lw=1.2, label=f"Mean growth {growth_df['pct_growth'].mean():.0f}%")
        for bar, val in zip(bars, growth_df['pct_growth']):
            ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, f'{val:+.0f}%', va='center', fontsize=6.5)
        patches = [mpatches.Patch(color=v,label=k) for k,v in region_colors.items()]
        ax.legend(handles=patches+[plt.Line2D([0],[0],color='red',ls='--',label=f"Mean growth {growth_df['pct_growth'].mean():.0f}%")], fontsize=8, loc='lower right')
        ax.set_xlabel('Download Speed Growth (%)')
        ax.set_title('Broadband Download Speed Growth\n2023 Q1 → 2025 Q4 by Province', fontsize=12)
        plt.tight_layout()
        plt.savefig('../../../outputs/ndt7/philippines/20_growth.png', dpi=150, bbox_inches='tight')
        plt.show()

### 15. Regional Growth Index — Trajectory Shape (Normalized to First Reliable Quarter = 100)
Complements the endpoint-only growth chart above (Q1 &#8594; Q4 % change) by showing the full trajectory per region, each region's own first valid quarter rebased to 100. Regions need at least half the available quarters with reliable data to be plotted, so noisy short-history regions are dropped instead of shown misleadingly. Rolled up from province to region level — adapted from the Myanmar/Laos NDT7 notebooks' growth-index cell, which plots at province level directly since those countries have far fewer top-level admin units than the province counts tracked here.


In [ ]:
if not HAS_RELIABLE_DATA:
    print("No reliable data for Broadband — skipping regional growth-index plot.")
else:
    quarter_order = sorted(master['label'].unique())
    MIN_Q = max(3, len(quarter_order) // 2)

    region_q = (
        master.groupby(['region', 'label'])
        .apply(lambda g: pd.Series({
            'avg_d_mbps': np.average(g['avg_d_mbps'], weights=g['total_tests']),
            'avg_u_mbps': np.average(g['avg_u_mbps'], weights=g['total_tests']),
            'avg_lat_ms_wt': np.average(g['avg_lat_ms_wt'], weights=g['total_tests']),
        }), include_groups=False)
        .reset_index()
    )

    metrics = [('avg_d_mbps', 'Download (Mbps)'), ('avg_u_mbps', 'Upload (Mbps)'), ('avg_lat_ms_wt', 'Latency (ms)')]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
    any_plotted = False
    for ax, (col, label) in zip(axes, metrics):
        piv = region_q.pivot(index='region', columns='label', values=col).reindex(columns=quarter_order)
        valid = piv.notna().sum(axis=1)
        keep = valid[valid >= MIN_Q].index.tolist()
        if not keep:
            ax.text(0.5, 0.5, 'not enough data', transform=ax.transAxes, ha='center')
            continue
        for reg in keep:
            s = piv.loc[reg].dropna()
            xs = [quarter_order.index(q) for q in s.index]
            ax.plot(xs, s.values / s.iloc[0] * 100, marker='o', ms=3, lw=1.3,
                    color=region_colors.get(reg, 'gray'), label=reg)
            any_plotted = True
        ax.axhline(100, color='grey', ls='--', lw=1)
        step = max(1, len(quarter_order) // 6)
        ax.set_xticks(range(0, len(quarter_order), step))
        ax.set_xticklabels([quarter_order[i] for i in range(0, len(quarter_order), step)], rotation=45, ha='right', fontsize=8)
        ax.set_title(f'{label}\n(region: own 1st valid quarter = 100)', fontsize=10)
    axes[0].set_ylabel('Growth index')
    if any_plotted:
        axes[0].legend(fontsize=7, ncol=2, loc='upper left')
    plt.suptitle(f'Broadband Regional Growth Index (regions with >= {MIN_Q} valid quarters)', fontsize=13)
    plt.tight_layout()
    plt.savefig('../../../outputs/ndt7/philippines/21_growth_index_regional.png', dpi=150, bbox_inches='tight')
    plt.show()


### 15.5 Measured Speed Choropleth — Actual NDT7 Download Speed by Province
The "Internet Tier" map earlier in this notebook is a GDP-based *expectation*. This map is the actual measured NDT7 average, on the same province geometry, so the two can be compared directly. Adapted from the Myanmar/Laos NDT7 notebooks' choropleth cell (re-based here on `master`'s aggregated province averages instead of a live DuckDB query, since this pipeline works off the pre-aggregated CSV export, not the raw parquet).


In [ ]:
if not HAS_RELIABLE_DATA:
    print("No reliable data for Broadband — skipping measured-speed choropleth.")
else:
    prov_speed = (
        master.groupby('name')
        .agg(avg_d_mbps=('avg_d_mbps', 'mean'), total_tests=('total_tests', 'sum'))
        .reset_index()
    )
    geo_speed = thailand_gdf.merge(prov_speed, on='name', how='left')

    fig, ax = plt.subplots(figsize=(10, 12))
    geo_speed.plot(
        column='avg_d_mbps', ax=ax, cmap='YlGnBu', legend=True,
        edgecolor='white', linewidth=0.5,
        legend_kwds={'shrink': 0.5, 'label': 'Mean Download (Mbps)'},
        missing_kwds={'color': 'lightgrey', 'label': 'no reliable data'},
    )
    ax.set_title('Measured Broadband Download Speed by Province\n(actual NDT7 data, averaged across reliable quarters — not the GDP-based tier assumption map above)', fontsize=12)
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig('../../../outputs/ndt7/philippines/22_speed_choropleth.png', dpi=150, bbox_inches='tight')
    plt.show()


### 16. City Focus — Capital + Top Cities by Test Volume (Broadband)
**Skipped for Philippines** — the raw NDT7 parquet has no `city` column (unlike Cambodia/Myanmar/Laos),
so this section cannot be built the same way. The capital-vs-rest-of-country comparison is
covered in much greater depth by the PH-specific **"Manila vs Rest of PH"** section appended at
the end of this notebook (national/regional split, top-10/bottom-10 provinces, mean AND median).


In [ ]:
print("City Focus skipped for Philippines — no `city` column in the raw NDT7 data. "
      "See the PH-specific \'Manila vs Rest of PH\' section at the end of this notebook instead.")


### 17. Per-ISP Ranking — Top ISPs by Test Volume (Broadband)
Top 15 ISPs (min 100 tests) ranked by average download speed, bar length shows speed and
label shows test volume — adapted from the Myanmar/Laos NDT7 notebooks' per-ISP section.

In [ ]:
_isp_tbl = _con.execute(f"""
    SELECT isp, COUNT(*) AS n,
           AVG(mean_throughput_mbps) FILTER (WHERE type='download') AS avg_d_mbps
    FROM read_parquet('{RAW_ISP_PARQUET}')
    WHERE network_type = 'broadband' AND isp IS NOT NULL
    GROUP BY isp
    HAVING COUNT(*) >= 100
    ORDER BY n DESC
    LIMIT 15
""").df()

if _isp_tbl.empty:
    print("No ISP meets the n>=100 test threshold for Broadband — skipping per-ISP ranking.")
else:
    _isp_tbl = _isp_tbl.sort_values('avg_d_mbps', ascending=True)
    fig, ax = plt.subplots(figsize=(10, max(4, 0.4*len(_isp_tbl))))
    ax.barh(_isp_tbl['isp'].str.slice(0, 30), _isp_tbl['avg_d_mbps'], color='#457b9d', alpha=0.85)
    for yi, (_, row) in enumerate(_isp_tbl.iterrows()):
        ax.text(row['avg_d_mbps'] + 1, yi, f"{row['avg_d_mbps']:.0f} (n={row['n']:,})", va='center', fontsize=8)
    ax.set_xlabel('Avg Download (Mbps)')
    ax.set_title('Philippines — Top ISPs by Test Volume (Broadband)')
    plt.tight_layout()
    plt.savefig('../../../outputs/ndt7/philippines/24_per_isp.png', dpi=150, bbox_inches='tight')
    plt.show()

### 18. Market Share / HHI — ISP Concentration (Broadband)
Top 8 ISPs + "Others" + "(unknown ISP)" by share of tests. HHI (Herfindahl-Hirschman Index):
<1500 = competitive, 1500-2500 = moderate concentration, >2500 = highly concentrated —
adapted from the Myanmar/Laos NDT7 notebooks' market-share section.

In [ ]:
_mkt = _con.execute(f"""
    SELECT COALESCE(isp, '(unknown)') AS isp, COUNT(*) AS n
    FROM read_parquet('{RAW_ISP_PARQUET}')
    WHERE network_type = 'broadband'
    GROUP BY 1 ORDER BY n DESC
""").df()

if _mkt.empty:
    print("No Broadband data — skipping market share.")
else:
    _total = _mkt['n'].sum()
    _mkt['share'] = 100 * _mkt['n'] / _total
    _named = _mkt[_mkt['isp'] != '(unknown)'].reset_index(drop=True)
    _unknown_share = float(_mkt.loc[_mkt['isp'] == '(unknown)', 'share'].sum())
    TOPN = 8
    _top = _named.head(TOPN)
    _others_share = float(_named['share'].iloc[TOPN:].sum())
    _hhi = float((_named['share'] ** 2).sum())

    _rows = list(zip(_top['isp'].str.slice(0, 34), _top['share']))
    if _others_share > 0:
        _rows.append((f"Others ({len(_named) - len(_top)} ISPs)", _others_share))
    if _unknown_share > 0:
        _rows.append(("(unknown ISP)", _unknown_share))
    _labels = [r[0] for r in _rows][::-1]
    _vals = [r[1] for r in _rows][::-1]
    _colors = ['#adb5bd' if l.startswith('Others') else '#ced4da' if l.startswith('(unknown') else '#457b9d' for l in _labels]

    fig, ax = plt.subplots(figsize=(10, max(4, 0.45 * len(_vals))))
    y = list(range(len(_vals)))
    ax.barh(y, _vals, color=_colors, edgecolor='white', linewidth=0.5)
    ax.set_yticks(y)
    ax.set_yticklabels(_labels, fontsize=9)
    for yi, v in zip(y, _vals):
        ax.text(v + max(_vals) * 0.01, yi, f"{v:.1f}%", va='center', fontsize=8)
    ax.set_xlabel('Market share (% of tests)')
    ax.set_title(f"Philippines — ISP Market Share (Broadband) \u00b7 HHI={_hhi:.0f}")
    plt.tight_layout()
    plt.savefig('../../../outputs/ndt7/philippines/25_market_share.png', dpi=150, bbox_inches='tight')
    plt.show()

    _hhi_note = 'competitive' if _hhi < 1500 else ('moderate' if _hhi < 2500 else 'highly concentrated')
    print(f"HHI={_hhi:.0f} ({_hhi_note}) | Top5 share={_named['share'].head(5).sum():.1f}% | unknown={_unknown_share:.1f}%")

### 19. EDA Key Findings Summary

## Key Findings from EDA

**Pending first execution** — this notebook has not been run against
`data/ph/mlab_ph_clean.parquet` yet (not available locally in this repo). Once run, replace this
placeholder with: overall `is_reliable` rate (total_tests>=100 basis, not tile-based), which
regions clear/miss the bar, GDP-vs-speed correlation strength (if enough reliable regions),
top/bottom regions by download speed, and how the region-level picture here compares to the
richer province/island-level findings in the PH-specific sections below.


## Export for Comparison Notebook
*Saves province × quarter data to CSV.*

In [ ]:
import os
os.makedirs("../../../data/exports", exist_ok=True)

EXPORT_COLS = [
    "name", "label", "year", "quarter",
    "avg_d_mbps", "avg_u_mbps", "avg_lat_ms_wt",
    "total_tests", "n_tiles", "is_reliable",
    "region", "internet_tier", "pop_2024",
    "gdp_per_capita_thb_2021", "density_per_km2",
]
existing_cols = [c for c in EXPORT_COLS if c in master.columns]
out = master[existing_cols].copy()
out = out.rename(columns={"name": "province", "label": "quarter"})

PATH = "../../../data/exports/ndt7_broadband_philippines_reliable_province_quarterly.csv"
out.to_csv(PATH, index=False)
print(f"Exported {len(out)} rows (reliable-only) -> {PATH}")
if len(out):
    print(out.head(3).to_string())
else:
    print("(0 rows — reliable-only export is empty; use the un-filtered prep-notebook output "
          "'ndt7_philippines_province_quarterly.csv' for any downstream use that needs raw coverage instead of the reliable subset.)")

---
# Part 2 — Mobile/Cellular

## Full EDA — NDT7 Mobile/Cellular, Philippines (2023 Q1 – 2025 Q4)

### 1. Load Province x Quarter Data

In [ ]:
import warnings
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from scipy import stats
warnings.filterwarnings('ignore')

NDT7_CSV = '../../../data/exports/ndt7_mobile_philippines_province_quarterly.csv'
GEOJSON  = '../../../data/geo/philippines_provinces.geojson'
PROV_REF = '../../../data/reference/philippines_reference.csv'

Q_TO_MONTH = {1: 1, 2: 4, 3: 7, 4: 10}

thailand_gdf = gpd.read_file(GEOJSON).to_crs(4326)

master = pd.read_csv(NDT7_CSV)
master = master.rename(columns={'province': 'name', 'quarter': 'label', 'quarter.1': 'quarter'})
master['month'] = master['quarter'].map(Q_TO_MONTH)
master['period_start'] = pd.to_datetime(dict(year=master['year'], month=master['month'], day=1))

prov_ref = pd.read_csv(PROV_REF)
prov_ref['gdp_per_capita_usd'] = prov_ref['gdp_per_capita_thb_2021']

print(f"Master shape: {master.shape}")
print(f"Date range: {master['period_start'].min().date()} -> {master['period_start'].max().date()}")
print(f"Quarters:   {sorted(master['label'].unique())}")
print(f"Provinces:  {master['name'].nunique()} (of {len(prov_ref)} in reference — NDT7 has zero test volume for the rest)")

In [ ]:
# is_reliable already computed upstream (philippines_ndt7_prep.ipynb) —
# NOTE: unlike KH/TH/VN, Philippines' is_reliable is `total_tests >= 100` ONLY (no tile-binning,
# no n_tiles — see prep notebook intro). Not directly comparable to the other tigger countries'
# `n_tiles>=5` bar; treat PH's reliability rate as its own methodology.
reliable_count = master.groupby('label')['is_reliable'].sum()
print('Reliable region-quarters per label:')
print(reliable_count.to_string())
print(f"\nOverall reliable: {master['is_reliable'].sum()} / {len(master)} ({master['is_reliable'].mean():.1%})")


### 2. Data Quality — Coverage per Province

In [ ]:
expected_n = len(prov_ref)
expected = expected_n * master['label'].nunique()
print(f"Expected rows (if every province had every quarter): {expected} | Actual: {len(master)} | Missing: {expected - len(master)}")

all_provinces = set(thailand_gdf['name'])
provinces_with_data = set(master['name'].unique())
zero_coverage = sorted(all_provinces - provinces_with_data)
if zero_coverage:
    print(f"\nProvinces with ZERO NDT7 test volume in any quarter ({len(zero_coverage)}): {zero_coverage}")
else:
    print("\nEvery province has at least some NDT7 data in at least one quarter.")

low_test = master[master['total_tests'] < 100][['name','label','total_tests','n_tiles']].sort_values('total_tests')
print(f"\nLow-test quarters (< 100 tests) — {len(low_test)} of {len(master)} province-quarter rows")
print(low_test.head(15).to_string(index=False))

In [ ]:
# Drop unreliable province-quarters before any stats/plots use master downstream — same rule as every
# other country/source. For Philippines this empties master entirely (0 reliable rows); sections below
# report that explicitly rather than silently rendering blank charts.
master = master[master['is_reliable']].copy()
print(f"Filtered to reliable province-quarters: {len(master)} rows kept")
HAS_RELIABLE_DATA = len(master) > 0
if not HAS_RELIABLE_DATA:
    print("\nNo province-quarters clear the reliability threshold — sections 3-14 below will report "
          "'no reliable data' instead of plotting.")

### 3. Speed Distribution — All Provinces, All Quarters

In [ ]:
if not HAS_RELIABLE_DATA:
    print("No reliable data for Mobile/Cellular — skipping speed-distribution plot.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].hist(master['avg_d_mbps'].dropna(), bins=60, color='#457b9d', edgecolor='none', alpha=0.85)
    axes[0].axvline(master['avg_d_mbps'].median(), color='red', ls='--', lw=1.5, label=f'Median {master["avg_d_mbps"].median():.0f} Mbps')
    axes[0].set_xlabel('Avg Download (Mbps)'); axes[0].set_title('Download Speed Distribution'); axes[0].legend()

    axes[1].hist(master['avg_u_mbps'].dropna(), bins=60, color='#2a9d8f', edgecolor='none', alpha=0.85)
    axes[1].axvline(master['avg_u_mbps'].median(), color='red', ls='--', lw=1.5, label=f'Median {master["avg_u_mbps"].median():.0f} Mbps')
    axes[1].set_xlabel('Avg Upload (Mbps)'); axes[1].set_title('Upload Speed Distribution'); axes[1].legend()

    axes[2].hist(master['avg_lat_ms_wt'].dropna(), bins=60, color='#e76f51', edgecolor='none', alpha=0.85)
    axes[2].axvline(master['avg_lat_ms_wt'].median(), color='navy', ls='--', lw=1.5, label=f'Median {master["avg_lat_ms_wt"].median():.0f} ms')
    axes[2].set_xlabel('Avg Latency (ms)'); axes[2].set_title('Latency Distribution'); axes[2].legend()

    plt.suptitle('Mobile/Cellular Metric Distributions — All Provinces, All Quarters (2023–2025)', fontsize=13)
    plt.tight_layout()

    plt.savefig('../../../outputs/ndt7/philippines/mobile/11_upload_speed_dist.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(master[['avg_d_mbps','avg_u_mbps','avg_lat_ms_wt','total_tests']].describe().round(2))

### 4. Province Ranking — Mean Download Speed (All Quarters)

In [ ]:
region_colors = {
    'Luzon': '#1f77b4',
    'Visayas': '#ff7f0e',
    'Mindanao': '#2ca02c',
}

if not HAS_RELIABLE_DATA:
    prov_mean = pd.DataFrame(columns=['name','region','mean_dl','mean_ul','mean_lat','mean_tests'])
    print("No reliable data for Mobile/Cellular — skipping province-ranking plot.")
else:
    prov_mean = (
        master.groupby(['name','region'])
        .agg(
            mean_dl=('avg_d_mbps','mean'),
            mean_ul=('avg_u_mbps','mean'),
            mean_lat=('avg_lat_ms_wt','mean'),
            mean_tests=('total_tests','mean'),
        )
        .reset_index()
        .sort_values('mean_dl', ascending=True)
    )

    fig, ax = plt.subplots(figsize=(14, 22))
    colors = [region_colors.get(r,'gray') for r in prov_mean['region']]
    bars = ax.barh(prov_mean['name'], prov_mean['mean_dl'], color=colors, edgecolor='none')
    for bar, val in zip(bars, prov_mean['mean_dl']):
        ax.text(bar.get_width() + 1.5, bar.get_y() + bar.get_height()/2, f'{val:.0f}', va='center', fontsize=7)
    ax.axvline(prov_mean['mean_dl'].mean(), color='black', ls='--', lw=1, label=f'National mean {prov_mean["mean_dl"].mean():.0f} Mbps')
    patches = [mpatches.Patch(color=v, label=k) for k, v in region_colors.items()]
    ax.legend(handles=patches, loc='lower right', fontsize=9)
    ax.set_xlabel('Mean Download Speed (Mbps)')
    ax.set_title('Mobile/Cellular Download Speed by Province\n(weighted avg per quarter, averaged 2023 Q1 – 2025 Q4)', fontsize=12)
    plt.tight_layout()
    plt.savefig('../../../outputs/ndt7/philippines/mobile/12_province_ranking_mean.png', dpi=150, bbox_inches='tight')
    plt.show()

### 5. Time Series — Download Speed Heatmap (Province × Quarter)

In [ ]:
if not HAS_RELIABLE_DATA:
    print("No reliable data for Mobile/Cellular — skipping heatmap.")
else:
    pivot_dl = master.pivot_table(index='name', columns='period_start', values='avg_d_mbps')
    pivot_dl.columns = [pd.Timestamp(c).strftime('%Y-Q') + str((pd.Timestamp(c).month-1)//3+1) for c in pivot_dl.columns]
    pivot_dl = pivot_dl.loc[prov_mean.sort_values('mean_dl', ascending=False)['name']]

    fig, ax = plt.subplots(figsize=(16, 22))
    im = ax.imshow(pivot_dl.values, aspect='auto', cmap='YlOrRd', interpolation='nearest')
    plt.colorbar(im, ax=ax, label='Avg Download (Mbps)', shrink=0.5)
    ax.set_xticks(range(len(pivot_dl.columns)))
    ax.set_xticklabels(pivot_dl.columns, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(len(pivot_dl.index)))
    ax.set_yticklabels(pivot_dl.index, fontsize=8)
    ax.set_title('Mobile/Cellular Download Speed Heatmap\n(province × quarter, sorted by mean speed)', fontsize=12)
    plt.tight_layout()
    plt.savefig('../../../outputs/ndt7/philippines/mobile/13_heatmap_quarterly.png', dpi=150, bbox_inches='tight')
    plt.show()

### 6. Regional Comparison — Download, Upload, Latency

In [ ]:
region_order = ['Luzon', 'Visayas', 'Mindanao']

if not HAS_RELIABLE_DATA:
    print("No reliable data for Mobile/Cellular — skipping regional boxplot.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    for ax, metric, label, color_key in [
        (axes[0], 'avg_d_mbps',   'Download (Mbps)', '#457b9d'),
        (axes[1], 'avg_u_mbps',   'Upload (Mbps)',   '#2a9d8f'),
        (axes[2], 'avg_lat_ms_wt','Latency (ms)',    '#e76f51'),
    ]:
        data = [master[master['region']==r][metric].dropna().values for r in region_order]
        bp = ax.boxplot(data, patch_artist=True, notch=False, medianprops=dict(color='white', linewidth=2))
        for patch, region in zip(bp['boxes'], region_order):
            patch.set_facecolor(region_colors[region])
            patch.set_alpha(0.85)
        ax.set_xticklabels(region_order, rotation=30, ha='right', fontsize=8)
        ax.set_ylabel(label)
        ax.set_title(f'{label} by Region')
    plt.suptitle('Mobile/Cellular Performance by Region — All Quarters (2023–2025)', fontsize=13)
    plt.tight_layout()
    plt.savefig('../../../outputs/ndt7/philippines/mobile/14_regional_boxplot.png', dpi=150, bbox_inches='tight')
    plt.show()

### 7. Download Speed Trend — Top, Bottom & Interesting Provinces

In [ ]:
if not HAS_RELIABLE_DATA:
    print("No reliable data for Mobile/Cellular — skipping time-series trend plot.")
else:
    latest_q = master['label'].max()
    latest = master[master['label'] == latest_q]
    top5 = latest.nlargest(5, 'avg_d_mbps')['name'].tolist()
    bot5 = latest.nsmallest(5, 'avg_d_mbps')['name'].tolist()

    fig, axes = plt.subplots(2, 1, figsize=(16, 12), sharex=True)
    groups = [
        (f'Top 5 Fastest ({latest_q})', top5, '#e63946'),
        (f'Bottom 5 Slowest ({latest_q})', bot5, '#6d6875'),
    ]
    for ax, (title, provinces, base_color) in zip(axes, groups):
        for prov in provinces:
            df_p = master[master['name']==prov].sort_values('label')
            if df_p.empty: continue
            ax.plot(df_p['period_start'], df_p['avg_d_mbps'], marker='o', markersize=5, label=prov, linewidth=1.8)
        ax.set_ylabel('Avg Download (Mbps)')
        ax.set_title(title)
        ax.legend(fontsize=8, loc='upper left')
        ax.grid(axis='y', alpha=0.3)
        ax.tick_params(axis='x', rotation=30)
    plt.suptitle('Download Speed Trend (2023 Q1 – 2025 Q4)', fontsize=13)
    plt.tight_layout()
    plt.savefig('../../../outputs/ndt7/philippines/mobile/15_time_series.png', dpi=150, bbox_inches='tight')
    plt.show()

### 8. GDP per Capita vs Download Speed — Does Money = Speed?

In [ ]:
if not HAS_RELIABLE_DATA:
    corr_df = pd.DataFrame(columns=['name','region','mean_dl','gdp_per_capita_thb_2021','internet_tier','pop_2024','density_per_km2'])
    r, p, r2, p2 = float('nan'), float('nan'), float('nan'), float('nan')
    print("No reliable data for Mobile/Cellular — skipping GDP/density correlation plot.")
else:
    corr_df = prov_mean.merge(
        prov_ref[['province_en','gdp_per_capita_thb_2021','internet_tier','pop_2024','density_per_km2']].rename(columns={'province_en':'name'}),
        on='name', how='left'
    )

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    for _, row in corr_df.iterrows():
        c = region_colors.get(row['region'], 'gray')
        axes[0].scatter(row['gdp_per_capita_thb_2021']/1000, row['mean_dl'],
                        color=c, s=row['pop_2024']/80000, alpha=0.7, edgecolors='white', lw=0.5)
        if row['gdp_per_capita_thb_2021'] > corr_df['gdp_per_capita_thb_2021'].quantile(0.9) or row['mean_dl'] > corr_df['mean_dl'].quantile(0.9) or row['mean_dl'] < corr_df['mean_dl'].quantile(0.1):
            axes[0].annotate(row['name'], (row['gdp_per_capita_thb_2021']/1000, row['mean_dl']), fontsize=6.5, ha='left', va='bottom')

    x = corr_df['gdp_per_capita_thb_2021'].dropna()/1000
    y = corr_df.loc[x.index,'mean_dl']
    if x.nunique() > 1:
        slope, intercept, r, p, _ = stats.linregress(x, y)
        xline = np.linspace(x.min(), x.max(), 100)
        axes[0].plot(xline, slope*xline+intercept, 'k--', lw=1.5, label=f'OLS (r={r:.2f}, p={p:.3f})')
        ols_handle = [plt.Line2D([0],[0],color='k',ls='--',label=f'OLS r={r:.2f}')]
    else:
        r, p = float('nan'), float('nan')
        axes[0].text(0.5, 0.05, 'GDP per capita uniform across regions — no regression', transform=axes[0].transAxes, ha='center', fontsize=8, style='italic')
        ols_handle = []
    axes[0].set_xlabel('GDP per Capita (000s USD, 2023)')
    axes[0].set_ylabel('Mean Download Speed (Mbps)')
    axes[0].set_title('GDP per Capita vs Avg Download Speed')
    patches = [mpatches.Patch(color=v, label=k) for k,v in region_colors.items()]
    axes[0].legend(handles=patches+ols_handle, fontsize=7)

    for _, row in corr_df.iterrows():
        c = region_colors.get(row['region'],'gray')
        axes[1].scatter(row['density_per_km2'], row['mean_dl'], color=c, s=row['pop_2024']/80000, alpha=0.7, edgecolors='white', lw=0.5)
        if row['density_per_km2'] > corr_df['density_per_km2'].quantile(0.9) or row['mean_dl'] > corr_df['mean_dl'].quantile(0.9) or row['mean_dl'] < corr_df['mean_dl'].quantile(0.1):
            axes[1].annotate(row['name'], (row['density_per_km2'], row['mean_dl']), fontsize=6.5, ha='left', va='bottom')

    x2 = corr_df['density_per_km2'].dropna()
    y2 = corr_df.loc[x2.index,'mean_dl']
    if x2.nunique() > 1:
        s2,i2,r2,p2,_ = stats.linregress(np.log1p(x2), y2)
        xline2 = np.linspace(x2.min(), x2.max(), 200)
        axes[1].plot(xline2, s2*np.log1p(xline2)+i2, 'k--', lw=1.5, label=f'OLS log (r={r2:.2f}, p={p2:.3f})')
    else:
        r2, p2 = float('nan'), float('nan')
    axes[1].set_xlabel('Population Density (per km²)')
    axes[1].set_xscale('log')
    axes[1].set_ylabel('Mean Download Speed (Mbps)')
    axes[1].set_title('Population Density vs Avg Download Speed (log scale)')
    axes[1].legend(fontsize=7)

    plt.suptitle('Economic & Demographic Correlates of Mobile/Cellular Performance', fontsize=13)
    plt.tight_layout()
    plt.savefig('../../../outputs/ndt7/philippines/mobile/16_gdp_speed_scatter.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Pearson r (GDP vs download):     {r:.3f}  p={p:.4f}")
    print(f"Pearson r (log density vs download): {r2:.3f}  p={p2:.4f}")

### 8.5 Multivariate Model — GDP + Density + Tier + Region Together

In [ ]:
import statsmodels.formula.api as smf

if not HAS_RELIABLE_DATA or len(corr_df.dropna(subset=['mean_dl'])) < 5:
    print("No reliable data for Mobile/Cellular — skipping multivariate model (need >=5 observations to fit).")
else:
    model_df = corr_df.dropna(subset=['mean_dl', 'gdp_per_capita_thb_2021', 'density_per_km2', 'region', 'internet_tier']).copy()
    model_df['log_gdp'] = np.log(model_df['gdp_per_capita_thb_2021'])
    model_df['log_density'] = np.log1p(model_df['density_per_km2'])

    terms = []
    if model_df['log_gdp'].nunique() > 1:
        terms.append('log_gdp')
    if model_df['log_density'].nunique() > 1:
        terms.append('log_density')
    if model_df['internet_tier'].nunique() > 1:
        terms.append("C(internet_tier)")
    if model_df['region'].nunique() > 1:
        terms.append('C(region)')

    if not terms:
        print("No varying predictors — skipping multivariate model.")
    else:
        formula = 'mean_dl ~ ' + ' + '.join(terms)
        mv_model = smf.ols(formula, data=model_df).fit()
        print(mv_model.summary())
        print(f"\nMultivariate R\u00b2 = {mv_model.rsquared:.3f}  (bivariate GDP-only r\u00b2 = {r**2:.3f})")
        print(f"Adjusted R\u00b2 = {mv_model.rsquared_adj:.3f}")

### 9. Upload/Download Ratio — Symmetry Check

In [ ]:
if not HAS_RELIABLE_DATA:
    print("No reliable data for Mobile/Cellular — skipping UL/DL ratio plot.")
else:
    master['ul_dl_ratio'] = master['avg_u_mbps'] / master['avg_d_mbps']
    ratio_mean = master.groupby('name')['ul_dl_ratio'].mean().sort_values(ascending=False).reset_index()
    ratio_mean = ratio_mean.merge(prov_ref[['province_en','region']].rename(columns={'province_en':'name'}), on='name', how='left')

    fig, ax = plt.subplots(figsize=(14, 20))
    colors = [region_colors.get(r,'gray') for r in ratio_mean['region']]
    ax.barh(ratio_mean['name'], ratio_mean['ul_dl_ratio'], color=colors, alpha=0.85)
    ax.axvline(1.0, color='red', ls='--', lw=1.5, label='Symmetric (1:1)')
    ax.axvline(ratio_mean['ul_dl_ratio'].mean(), color='black', ls=':', lw=1, label=f'Mean ratio {ratio_mean["ul_dl_ratio"].mean():.2f}')
    ax.set_xlabel('Upload / Download Ratio')
    ax.set_title('Upload/Download Ratio by Province\n(mobile: DL-heavy expected; high ratio = 5G or symmetric cell)')
    patches = [mpatches.Patch(color=v, label=k) for k,v in region_colors.items()]
    ax.legend(handles=patches+[
        plt.Line2D([0],[0],color='red',ls='--',label='Symmetric 1:1'),
        plt.Line2D([0],[0],color='black',ls=':',label=f'Mean {ratio_mean["ul_dl_ratio"].mean():.2f}')
    ], fontsize=8, loc='lower right')
    plt.tight_layout()
    plt.savefig('../../../outputs/ndt7/philippines/mobile/17_uldl_ratio_bar.png', dpi=150, bbox_inches='tight')
    plt.show()

    divergent_ratio = ratio_mean[ratio_mean['ul_dl_ratio'] > 0.5]
    print(f"Provinces with UL/DL ratio > 0.5 ({len(divergent_ratio)}):")
    print(divergent_ratio[['name','region','ul_dl_ratio']].to_string(index=False))

### 10. Province Summary — Tier & UL/DL Ratio

In [ ]:
from scipy import stats
import numpy as np

if not HAS_RELIABLE_DATA:
    high_ratio = pd.Series(dtype=float)
    prov_mean = pd.DataFrame(columns=['name','region','mean_dl','mean_ul','mean_lat','mean_tests','mean_ratio','tier'])
    print("No reliable data for Mobile/Cellular — skipping province summary.")
else:
    master['ul_dl_ratio'] = master['avg_u_mbps'] / master['avg_d_mbps']
    high_ratio = master.groupby('name')['ul_dl_ratio'].mean()

    prov_mean = (
        master.groupby(['name','region'])
        .agg(
            mean_dl=('avg_d_mbps','mean'),
            mean_ul=('avg_u_mbps','mean'),
            mean_lat=('avg_lat_ms_wt','mean'),
            mean_tests=('total_tests','mean'),
            mean_ratio=('ul_dl_ratio','mean'),
        )
        .reset_index()
        .sort_values('mean_dl', ascending=False)
    )
    prov_mean['tier'] = prov_mean['name'].map(prov_ref.set_index('province_en')['internet_tier'].to_dict())

### 12. Full Province Summary Table

In [ ]:
if not HAS_RELIABLE_DATA:
    print("No reliable data for Mobile/Cellular — nothing to summarize.")
else:
    summary = prov_mean.merge(
        prov_ref[['province_en','pop_2024','gdp_per_capita_thb_2021','internet_tier']].rename(columns={'province_en':'name'}),
        on='name', how='left'
    ).copy()
    summary['ul_dl_ratio'] = summary['name'].map(high_ratio).round(3)
    summary = summary.sort_values('mean_dl', ascending=False).round(2)
    print(summary[[
        'name','region','internet_tier','mean_dl','mean_ul','mean_lat','ul_dl_ratio','mean_tests','gdp_per_capita_thb_2021'
    ]].to_string(index=False))

### 13. UL/DL Symmetry — Fiber Penetration Evidence

In [ ]:
if not HAS_RELIABLE_DATA:
    print("No reliable data for Mobile/Cellular — skipping UL/DL symmetry deep-dive.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    ratio_sorted = prov_mean.sort_values('mean_ratio', ascending=True)
    colors = [region_colors.get(r,'gray') for r in ratio_sorted['region']]
    axes[0].barh(ratio_sorted['name'], ratio_sorted['mean_ratio'], color=colors, alpha=0.85)
    axes[0].axvline(prov_mean['mean_ratio'].mean(), color='black', ls='--', lw=1, label=f"National mean {prov_mean['mean_ratio'].mean():.2f}")
    axes[0].axvline(1.0, color='red', ls=':', lw=1, label='Symmetric 1:1')
    axes[0].set_xlabel('UL / DL Ratio')
    axes[0].set_title('UL/DL Symmetry Ratio by Province')
    patches = [mpatches.Patch(color=v,label=k) for k,v in region_colors.items()]
    axes[0].legend(handles=patches + [
        plt.Line2D([0],[0],color='black',ls='--',label=f"Mean {prov_mean['mean_ratio'].mean():.2f}"),
        plt.Line2D([0],[0],color='red',ls=':',label='Symmetric 1:1'),
    ], fontsize=7, loc='lower right')

    axes[1].scatter(prov_ref.set_index('province_en').loc[prov_mean['name'],'density_per_km2'].values,
                    prov_mean['mean_ratio'].values,
                    c=[region_colors.get(r,'gray') for r in prov_mean['region']],
                    s=80, alpha=0.7, edgecolors='white', lw=0.5)
    for _, row in prov_mean.iterrows():
        if row['mean_ratio'] > prov_mean['mean_ratio'].quantile(0.9) or row['mean_ratio'] < prov_mean['mean_ratio'].quantile(0.1):
            density = prov_ref.loc[prov_ref['province_en']==row['name'],'density_per_km2'].values
            if len(density):
                axes[1].annotate(row['name'], (density[0], row['mean_ratio']), fontsize=6.5, ha='left', va='bottom')
    axes[1].set_xscale('log')
    axes[1].set_xlabel('Population Density (per km², log scale)')
    axes[1].set_ylabel('UL/DL Ratio')
    axes[1].set_title('Density vs UL/DL Symmetry')
    axes[1].legend(handles=patches, fontsize=7)

    plt.suptitle('Upload/Download Symmetry — Mobile/Cellular', fontsize=13)
    plt.tight_layout()
    plt.savefig('../../../outputs/ndt7/philippines/mobile/19_uldl_symmetry_ratio.png', dpi=150, bbox_inches='tight')
    plt.show()

### 14. Temporal: Speed Growth 2023 Q1 → 2025 Q4

In [ ]:
if not HAS_RELIABLE_DATA:
    print("No reliable data for Mobile/Cellular — skipping temporal growth plot.")
else:
    first = master[master['label']=='2023-Q1'].set_index('name')['avg_d_mbps']
    last  = master[master['label']=='2025-Q4'].set_index('name')['avg_d_mbps']
    growth = ((last - first) / first * 100).dropna().sort_values(ascending=False)

    if growth.empty:
        print("No province has both a 2023-Q1 and 2025-Q4 reliable observation — skipping growth plot.")
    else:
        growth_df = growth.reset_index()
        growth_df.columns = ['name','pct_growth']
        growth_df = growth_df.merge(prov_ref[['province_en','region','internet_tier']].rename(columns={'province_en':'name'}), on='name', how='left')

        fig, ax = plt.subplots(figsize=(14, 20))
        colors = [region_colors.get(r,'gray') for r in growth_df['region']]
        bars = ax.barh(growth_df['name'], growth_df['pct_growth'], color=colors, alpha=0.85)
        ax.axvline(0, color='black', lw=0.8)
        ax.axvline(growth_df['pct_growth'].mean(), color='red', ls='--', lw=1.2, label=f"Mean growth {growth_df['pct_growth'].mean():.0f}%")
        for bar, val in zip(bars, growth_df['pct_growth']):
            ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, f'{val:+.0f}%', va='center', fontsize=6.5)
        patches = [mpatches.Patch(color=v,label=k) for k,v in region_colors.items()]
        ax.legend(handles=patches+[plt.Line2D([0],[0],color='red',ls='--',label=f"Mean growth {growth_df['pct_growth'].mean():.0f}%")], fontsize=8, loc='lower right')
        ax.set_xlabel('Download Speed Growth (%)')
        ax.set_title('Mobile/Cellular Download Speed Growth\n2023 Q1 → 2025 Q4 by Province', fontsize=12)
        plt.tight_layout()
        plt.savefig('../../../outputs/ndt7/philippines/mobile/20_growth.png', dpi=150, bbox_inches='tight')
        plt.show()

### 15. Regional Growth Index — Trajectory Shape (Normalized to First Reliable Quarter = 100)
Complements the endpoint-only growth chart above (Q1 &#8594; Q4 % change) by showing the full trajectory per region, each region's own first valid quarter rebased to 100. Regions need at least half the available quarters with reliable data to be plotted, so noisy short-history regions are dropped instead of shown misleadingly. Rolled up from province to region level — adapted from the Myanmar/Laos NDT7 notebooks' growth-index cell, which plots at province level directly since those countries have far fewer top-level admin units than the province counts tracked here.


In [ ]:
if not HAS_RELIABLE_DATA:
    print("No reliable data for Mobile/Cellular — skipping regional growth-index plot.")
else:
    quarter_order = sorted(master['label'].unique())
    MIN_Q = max(3, len(quarter_order) // 2)

    region_q = (
        master.groupby(['region', 'label'])
        .apply(lambda g: pd.Series({
            'avg_d_mbps': np.average(g['avg_d_mbps'], weights=g['total_tests']),
            'avg_u_mbps': np.average(g['avg_u_mbps'], weights=g['total_tests']),
            'avg_lat_ms_wt': np.average(g['avg_lat_ms_wt'], weights=g['total_tests']),
        }), include_groups=False)
        .reset_index()
    )

    metrics = [('avg_d_mbps', 'Download (Mbps)'), ('avg_u_mbps', 'Upload (Mbps)'), ('avg_lat_ms_wt', 'Latency (ms)')]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
    any_plotted = False
    for ax, (col, label) in zip(axes, metrics):
        piv = region_q.pivot(index='region', columns='label', values=col).reindex(columns=quarter_order)
        valid = piv.notna().sum(axis=1)
        keep = valid[valid >= MIN_Q].index.tolist()
        if not keep:
            ax.text(0.5, 0.5, 'not enough data', transform=ax.transAxes, ha='center')
            continue
        for reg in keep:
            s = piv.loc[reg].dropna()
            xs = [quarter_order.index(q) for q in s.index]
            ax.plot(xs, s.values / s.iloc[0] * 100, marker='o', ms=3, lw=1.3,
                    color=region_colors.get(reg, 'gray'), label=reg)
            any_plotted = True
        ax.axhline(100, color='grey', ls='--', lw=1)
        step = max(1, len(quarter_order) // 6)
        ax.set_xticks(range(0, len(quarter_order), step))
        ax.set_xticklabels([quarter_order[i] for i in range(0, len(quarter_order), step)], rotation=45, ha='right', fontsize=8)
        ax.set_title(f'{label}\n(region: own 1st valid quarter = 100)', fontsize=10)
    axes[0].set_ylabel('Growth index')
    if any_plotted:
        axes[0].legend(fontsize=7, ncol=2, loc='upper left')
    plt.suptitle(f'Mobile/Cellular Regional Growth Index (regions with >= {MIN_Q} valid quarters)', fontsize=13)
    plt.tight_layout()
    plt.savefig('../../../outputs/ndt7/philippines/mobile/21_growth_index_regional.png', dpi=150, bbox_inches='tight')
    plt.show()


### 15.5 Measured Speed Choropleth — Actual NDT7 Download Speed by Province
The "Internet Tier" map earlier in this notebook is a GDP-based *expectation*. This map is the actual measured NDT7 average, on the same province geometry, so the two can be compared directly. Adapted from the Myanmar/Laos NDT7 notebooks' choropleth cell (re-based here on `master`'s aggregated province averages instead of a live DuckDB query, since this pipeline works off the pre-aggregated CSV export, not the raw parquet).


In [ ]:
if not HAS_RELIABLE_DATA:
    print("No reliable data for Mobile/Cellular — skipping measured-speed choropleth.")
else:
    prov_speed = (
        master.groupby('name')
        .agg(avg_d_mbps=('avg_d_mbps', 'mean'), total_tests=('total_tests', 'sum'))
        .reset_index()
    )
    geo_speed = thailand_gdf.merge(prov_speed, on='name', how='left')

    fig, ax = plt.subplots(figsize=(10, 12))
    geo_speed.plot(
        column='avg_d_mbps', ax=ax, cmap='YlGnBu', legend=True,
        edgecolor='white', linewidth=0.5,
        legend_kwds={'shrink': 0.5, 'label': 'Mean Download (Mbps)'},
        missing_kwds={'color': 'lightgrey', 'label': 'no reliable data'},
    )
    ax.set_title('Measured Mobile/Cellular Download Speed by Province\n(actual NDT7 data, averaged across reliable quarters — not the GDP-based tier assumption map above)', fontsize=12)
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig('../../../outputs/ndt7/philippines/mobile/22_speed_choropleth.png', dpi=150, bbox_inches='tight')
    plt.show()


### 16. City Focus — Capital + Top Cities by Test Volume (Mobile/Cellular)
**Skipped for Philippines** — the raw NDT7 parquet has no `city` column (unlike Cambodia/Myanmar/Laos),
so this section cannot be built the same way. The capital-vs-rest-of-country comparison is
covered in much greater depth by the PH-specific **"Manila vs Rest of PH"** section appended at
the end of this notebook (national/regional split, top-10/bottom-10 provinces, mean AND median).


In [ ]:
print("City Focus skipped for Philippines — no `city` column in the raw NDT7 data. "
      "See the PH-specific \'Manila vs Rest of PH\' section at the end of this notebook instead.")


### 17. Per-ISP Ranking — Top ISPs by Test Volume (Mobile/Cellular)
Top 15 ISPs (min 100 tests) ranked by average download speed, bar length shows speed and
label shows test volume — adapted from the Myanmar/Laos NDT7 notebooks' per-ISP section.

In [ ]:
_isp_tbl = _con.execute(f"""
    SELECT isp, COUNT(*) AS n,
           AVG(mean_throughput_mbps) FILTER (WHERE type='download') AS avg_d_mbps
    FROM read_parquet('{RAW_ISP_PARQUET}')
    WHERE network_type = 'cellular' AND isp IS NOT NULL
    GROUP BY isp
    HAVING COUNT(*) >= 100
    ORDER BY n DESC
    LIMIT 15
""").df()

if _isp_tbl.empty:
    print("No ISP meets the n>=100 test threshold for Mobile/Cellular — skipping per-ISP ranking.")
else:
    _isp_tbl = _isp_tbl.sort_values('avg_d_mbps', ascending=True)
    fig, ax = plt.subplots(figsize=(10, max(4, 0.4*len(_isp_tbl))))
    ax.barh(_isp_tbl['isp'].str.slice(0, 30), _isp_tbl['avg_d_mbps'], color='#457b9d', alpha=0.85)
    for yi, (_, row) in enumerate(_isp_tbl.iterrows()):
        ax.text(row['avg_d_mbps'] + 1, yi, f"{row['avg_d_mbps']:.0f} (n={row['n']:,})", va='center', fontsize=8)
    ax.set_xlabel('Avg Download (Mbps)')
    ax.set_title('Philippines — Top ISPs by Test Volume (Mobile/Cellular)')
    plt.tight_layout()
    plt.savefig('../../../outputs/ndt7/philippines/mobile/24_per_isp.png', dpi=150, bbox_inches='tight')
    plt.show()

### 18. Market Share / HHI — ISP Concentration (Mobile/Cellular)
Top 8 ISPs + "Others" + "(unknown ISP)" by share of tests. HHI (Herfindahl-Hirschman Index):
<1500 = competitive, 1500-2500 = moderate concentration, >2500 = highly concentrated —
adapted from the Myanmar/Laos NDT7 notebooks' market-share section.

In [ ]:
_mkt = _con.execute(f"""
    SELECT COALESCE(isp, '(unknown)') AS isp, COUNT(*) AS n
    FROM read_parquet('{RAW_ISP_PARQUET}')
    WHERE network_type = 'cellular'
    GROUP BY 1 ORDER BY n DESC
""").df()

if _mkt.empty:
    print("No Mobile/Cellular data — skipping market share.")
else:
    _total = _mkt['n'].sum()
    _mkt['share'] = 100 * _mkt['n'] / _total
    _named = _mkt[_mkt['isp'] != '(unknown)'].reset_index(drop=True)
    _unknown_share = float(_mkt.loc[_mkt['isp'] == '(unknown)', 'share'].sum())
    TOPN = 8
    _top = _named.head(TOPN)
    _others_share = float(_named['share'].iloc[TOPN:].sum())
    _hhi = float((_named['share'] ** 2).sum())

    _rows = list(zip(_top['isp'].str.slice(0, 34), _top['share']))
    if _others_share > 0:
        _rows.append((f"Others ({len(_named) - len(_top)} ISPs)", _others_share))
    if _unknown_share > 0:
        _rows.append(("(unknown ISP)", _unknown_share))
    _labels = [r[0] for r in _rows][::-1]
    _vals = [r[1] for r in _rows][::-1]
    _colors = ['#adb5bd' if l.startswith('Others') else '#ced4da' if l.startswith('(unknown') else '#457b9d' for l in _labels]

    fig, ax = plt.subplots(figsize=(10, max(4, 0.45 * len(_vals))))
    y = list(range(len(_vals)))
    ax.barh(y, _vals, color=_colors, edgecolor='white', linewidth=0.5)
    ax.set_yticks(y)
    ax.set_yticklabels(_labels, fontsize=9)
    for yi, v in zip(y, _vals):
        ax.text(v + max(_vals) * 0.01, yi, f"{v:.1f}%", va='center', fontsize=8)
    ax.set_xlabel('Market share (% of tests)')
    ax.set_title(f"Philippines — ISP Market Share (Mobile/Cellular) \u00b7 HHI={_hhi:.0f}")
    plt.tight_layout()
    plt.savefig('../../../outputs/ndt7/philippines/mobile/25_market_share.png', dpi=150, bbox_inches='tight')
    plt.show()

    _hhi_note = 'competitive' if _hhi < 1500 else ('moderate' if _hhi < 2500 else 'highly concentrated')
    print(f"HHI={_hhi:.0f} ({_hhi_note}) | Top5 share={_named['share'].head(5).sum():.1f}% | unknown={_unknown_share:.1f}%")

### 19. EDA Key Findings Summary

## Key Findings from EDA

**Pending first execution** — this notebook has not been run against
`data/ph/mlab_ph_clean.parquet` yet (not available locally in this repo). Once run, replace this
placeholder with: overall `is_reliable` rate (total_tests>=100 basis, not tile-based), which
regions clear/miss the bar, GDP-vs-speed correlation strength (if enough reliable regions),
top/bottom regions by download speed, and how the region-level picture here compares to the
richer province/island-level findings in the PH-specific sections below.


## Export for Comparison Notebook
*Saves province × quarter data to CSV.*

In [ ]:
import os
os.makedirs("../../../data/exports", exist_ok=True)

EXPORT_COLS = [
    "name", "label", "year", "quarter",
    "avg_d_mbps", "avg_u_mbps", "avg_lat_ms_wt",
    "total_tests", "n_tiles", "is_reliable",
    "region", "internet_tier", "pop_2024",
    "gdp_per_capita_thb_2021", "density_per_km2",
]
existing_cols = [c for c in EXPORT_COLS if c in master.columns]
out = master[existing_cols].copy()
out = out.rename(columns={"name": "province", "label": "quarter"})

PATH = "../../../data/exports/ndt7_mobile_philippines_reliable_province_quarterly.csv"
out.to_csv(PATH, index=False)
print(f"Exported {len(out)} rows (reliable-only) -> {PATH}")
if len(out):
    print(out.head(3).to_string())
else:
    print("(0 rows — reliable-only export is empty; use the un-filtered prep-notebook output "
          "'ndt7_mobile_philippines_province_quarterly.csv' for any downstream use that needs raw coverage instead of the reliable subset.)")

---
# Part 3 — Philippines-Specific Additions

Everything below this point is **PH-specific content, kept per user decision rather than
dropped to force a 1:1 match with Cambodia/Thailand/Vietnam's tigger template** — the
Philippines' own source notebooks (`ndt7_ph_main.ipynb`, `ndt7_ph_manila.ipynb`,
`ndt7_ph_paper.ipynb`) have real analytical depth (island-group divide, ISP market share by
island, Manila-vs-rest-of-country drilldown, paper-support app-requirement analysis) that the
standard tigger template has no place for. Ported here close to verbatim from those three
already-executed notebooks (light edits only: adapted headings, one shared setup instead of
three separate ones where sections reuse `con`/`q`).

**Also not yet executed** — same caveat as the rest of this notebook.


## PH-Specific Setup

Shared DuckDB connection + helpers, reused by every PH-specific section below (island divide, ISP-by-island). Ported verbatim from `ndt7_ph_main.ipynb` Part 0.


In [ ]:
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import geopandas as gpd
from pathlib import Path

# --- resolve repo root regardless of the kernel's working dir ---
ROOT = Path.cwd()
while not (ROOT / 'data' / 'ph').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

CLEAN   = ROOT / 'data' / 'ph' / 'mlab_ph_clean.parquet'
GEOJSON = ROOT / 'data' / 'ph' / 'gadm41_PHL_1.geojson'
ADM_FIELD = 'NAME_1'          # GADM ADM1 province-name field (was 'shapeName' in the old ADM2 file)
assert CLEAN.exists(),   f'missing {CLEAN}'
assert GEOJSON.exists(), f'missing {GEOJSON}'

# --- one DuckDB connection; expose the cleaned parquet as a view called `t` ---
con = duckdb.connect()
con.execute("SET max_memory='12GB'")   # 32GB box; leaves headroom for pandas/geopandas
con.execute(f"SET temp_directory='{(ROOT / '.tmp').as_posix()}'")   # spill to disk instead of OOM on a 262M-row scan
# --- exclude one monitoring/bot client IP that fired 865k tests and dominated SultanKudarat
#     (59% of that province's fixed-broadband download tests). Diagnosis: ndt7_ph_manila Part 3. ---
FLOOD_IPS = ('58.69.220.245',)
_flood = ', '.join(f"'{ip}'" for ip in FLOOD_IPS)
con.execute(f"CREATE OR REPLACE VIEW t AS SELECT * FROM read_parquet('{CLEAN.as_posix()}') "
            f"WHERE client_ip NOT IN ({_flood})")

def q(sql):
    """Run SQL against the cleaned PH dataset (exposed as view `t`) and return a DataFrame."""
    return con.execute(sql).df()

# --- reusable SQL predicates = the analysis vocabulary ---
FIXED  = "network_type = 'broadband'"   # consumer fixed line
MOBILE = "network_type = 'cellular'"    # consumer mobile
DL     = "type = 'download'"
UL     = "type = 'upload'"

# --- plotting defaults + a consistent fixed/mobile color pair used throughout ---
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.grid': True, 'grid.alpha': 0.25,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11,
})
C_FIXED, C_MOBILE = '#0B6E5C', '#B8571F'

# --- sanity check ---
n_rows = con.execute("SELECT COUNT(*) FROM t").fetchone()[0]
print(f'ROOT    : {ROOT}')
print(f'CLEAN   : {CLEAN.name}  ({n_rows:,} rows)')
print(f'GEOJSON : {GEOJSON.name}')
print(f'excluded: {len(FLOOD_IPS)} flood IP(s) — {FLOOD_IPS}')                                                              

## Geographic mapping — province → region → island

v2 parquet มีคอลัมน์ `region` มาให้แล้ว (17 ภูมิภาค) จึง **ไม่ต้องสร้าง mapping จังหวัด→ภูมิภาคเองอีกต่อไป** —
เหลือแค่ชั้นบนสุด region→island group ที่ยังไม่มีในข้อมูล สร้างที่นี่ แล้ว register เป็นตาราง `reg_map`
(province, region, island) ใน DuckDB เพื่อให้ทุกส่วนถัดไป (Part 2 เกาะ, Part 3 region, Part 4 จังหวัด) join ใช้ร่วมกัน
· `assert` กันกรณีมี region ในข้อมูลที่หลุด ISLAND map

In [ ]:
from matplotlib.patches import Patch

# --- region -> island group. The v2 parquet already carries `region` (17 GADM-aligned codes),
# so only this top layer is built by hand; province->region is read straight from the data. ---
ISLAND = {r: 'Luzon' for r in ['NCR', 'CAR', 'Region I', 'Region II', 'Region III',
                               'Region IV-A', 'MIMAROPA', 'Region V']}
ISLAND.update({r: 'Visayas'  for r in ['Region VI', 'Region VII', 'Region VIII']})
ISLAND.update({r: 'Mindanao' for r in ['Region IX', 'Region X', 'Region XI', 'Region XII',
                                       'Region XIII', 'BARMM']})

reg_map = q("""
    SELECT DISTINCT province, region
    FROM t
    WHERE province IS NOT NULL AND region IS NOT NULL
""")
# fail loudly rather than silently dropping a region's tests in every downstream join
_missing = sorted(set(reg_map['region']) - set(ISLAND))
assert not _missing, f'region(s) not in the ISLAND map: {_missing}'
reg_map['island'] = reg_map['region'].map(ISLAND)
con.register('reg_map', reg_map)

ISLAND_COLORS = {'Luzon': '#1f77b4', 'Visayas': '#ff7f0e', 'Mindanao': '#2ca02c'}
print(f'reg_map: {len(reg_map)} provinces -> {reg_map.region.nunique()} regions -> {reg_map.island.nunique()} islands')
print(reg_map.groupby('island').province.count().to_string())


## PH-Specific Addition 1 — Island-Group Divide (North–South Digital Divide)

Ported verbatim from `ndt7_ph_main.ipynb` Part 2.


In [ ]:
island = q("""
    SELECT m.island,
           COUNT(*)                                                               AS tests,
           COUNT(DISTINCT t.province)                                             AS provinces,
           ROUND(100.0*SUM(CASE WHEN network_type='cellular' THEN 1 ELSE 0 END)/COUNT(*), 1) AS mobile_reliance_pct,
           ROUND(AVG(CASE    WHEN network_type='broadband' AND type='download' THEN mean_throughput_mbps END), 1) AS fixed_dl_mean,
           ROUND(MEDIAN(CASE WHEN network_type='broadband' AND type='download' THEN mean_throughput_mbps END), 1) AS fixed_dl_med,
           ROUND(AVG(CASE    WHEN network_type='cellular'  AND type='download' THEN mean_throughput_mbps END), 1) AS mobile_dl_mean,
           ROUND(MEDIAN(CASE WHEN network_type='cellular'  AND type='download' THEN mean_throughput_mbps END), 1) AS mobile_dl_med
    FROM t JOIN reg_map m ON t.province = m.province
    WHERE network_type IN ('broadband', 'cellular')
    GROUP BY 1
""").set_index('island').loc[['Luzon', 'Visayas', 'Mindanao']].reset_index()   # order North -> South

x, w = np.arange(3), 0.38
fig, (axm, axmd, axr) = plt.subplots(1, 3, figsize=(17, 5.4),
                                     gridspec_kw={'width_ratios': [2, 2, 1]})

# two speed panels (Mean / Median), fixed vs mobile — shared y so the mean > median skew is visible
for ax, fcol, mcol, ttl in [(axm, 'fixed_dl_mean', 'mobile_dl_mean', 'Mean'),
                            (axmd, 'fixed_dl_med',  'mobile_dl_med',  'Median')]:
    b1 = ax.bar(x - w/2, island[fcol], w, label='Fixed broadband', color=C_FIXED)
    b2 = ax.bar(x + w/2, island[mcol], w, label='Mobile',          color=C_MOBILE)
    ax.bar_label(b1, fmt='%.1f', padding=2)
    ax.bar_label(b2, fmt='%.1f', padding=2)
    ax.set_xticks(x)
    ax.set_xticklabels([f'{r.island}\n{r.tests/1e6:.0f}M · {r.provinces} prov' for _, r in island.iterrows()])
    ax.set_title(f'{ttl} download speed')
axm.set_ylabel('Download (Mbps)')
axmd.sharey(axm)
axm.legend()

# mobile reliance panel
bb = axr.bar(x, island['mobile_reliance_pct'], color='#6C5FA8')
axr.bar_label(bb, fmt='%.1f%%', padding=2)
axr.set_xticks(x)
axr.set_xticklabels(island['island'])
axr.set_ylabel('% of tests that are mobile')
axr.set_title('Mobile reliance')

fig.suptitle('PH North–South Digital Divide by Island Group (2023–2025)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

island

## PH-Specific Addition 2 — ISP Market Share by Island (Fixed Broadband)

Ported verbatim from `ndt7_ph_main.ipynb` Part 8 §4 (distinct from the standard §18 Market Share/HHI section earlier in this notebook, which is national-level, not split by island).


In [ ]:
sh = q("""
    SELECT m.island, t.isp, COUNT(*) AS tests
    FROM t JOIN reg_map m ON t.province = m.province
    WHERE network_type='broadband' AND isp IS NOT NULL
    GROUP BY 1, 2
""")
top_isps = sh.groupby('isp').tests.sum().sort_values(ascending=False).head(6).index.tolist()
sh['isp2'] = sh.isp.where(sh.isp.isin(top_isps), 'Other')
piv = sh.groupby(['island', 'isp2']).tests.sum().unstack(fill_value=0)
piv = piv.div(piv.sum(axis=1), axis=0) * 100
piv = piv.loc[['Luzon', 'Visayas', 'Mindanao']][ [c for c in top_isps if c in piv.columns] + ['Other'] ]

fig, ax = plt.subplots(figsize=(11, 6))
bottom = np.zeros(len(piv))
for i, c in enumerate(piv.columns):
    ax.bar(piv.index, piv[c], bottom=bottom,
           label=c[:26], color='#cccccc' if c == 'Other' else plt.cm.tab10(i))
    bottom += piv[c].values
ax.set_ylabel('% of fixed broadband tests')
ax.set_title('Fixed broadband ISP market share by island', fontweight='bold')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

piv.round(1)

## PH-Specific Addition 3 — Manila vs Rest of PH

Ported in full, verbatim, from `ndt7_ph_manila.ipynb` (13 cells) — opens its own DuckDB connection/view (separate from the setup cell above, exactly as the source notebook did) and defines its own `MANILA`/`AREA`/`MIN_TESTS` constants. This is the PH-specific answer to the standard template's §16 City Focus section, which was skipped earlier (no `city` column in the PH raw data).


## Part 0 · Setup

เหมือนเล่มหลักทุกอย่าง (path แบบไม่ขึ้นกับ working directory, DuckDB view ชื่อ `t`, helper `q()`, สี fixed/mobile)
บวกด้วยนิยาม `AREA` = Metro Manila / Rest of PH ที่ใช้ซ้ำทั้งเล่ม

In [ ]:
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# --- resolve repo root regardless of the kernel's working dir ---
ROOT = Path.cwd()
while not (ROOT / 'data' / 'ph').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

CLEAN = ROOT / 'data' / 'ph' / 'mlab_ph_clean.parquet'
assert CLEAN.exists(), f'missing {CLEAN}'

con = duckdb.connect()
con.execute("SET max_memory='12GB'")
con.execute(f"SET temp_directory='{(ROOT / '.tmp').as_posix()}'")   # spill to disk instead of OOM
# --- exclude one monitoring/bot client IP (865k tests, dominated SultanKudarat). Part 3 below
#     documents it and therefore reads the UNFILTERED `t_raw`; everything else uses filtered `t`. ---
FLOOD_IPS = ('58.69.220.245',)
_flood = ', '.join(f"'{ip}'" for ip in FLOOD_IPS)
con.execute(f"CREATE OR REPLACE VIEW t_raw AS SELECT * FROM read_parquet('{CLEAN.as_posix()}')")
con.execute(f"CREATE OR REPLACE VIEW t AS SELECT * FROM read_parquet('{CLEAN.as_posix()}') "
            f"WHERE client_ip NOT IN ({_flood})")

def q(sql):
    """Run SQL against the cleaned PH dataset (exposed as view `t`) and return a DataFrame."""
    return con.execute(sql).df()

# --- reusable SQL predicates = the analysis vocabulary (same as the main notebook) ---
FIXED  = "network_type = 'broadband'"
MOBILE = "network_type = 'cellular'"
DL     = "type = 'download'"
UL     = "type = 'upload'"

# --- the split this notebook is built around ---
MANILA = "MetropolitanManila"
AREA   = f"CASE WHEN province = '{MANILA}' THEN 'Metro Manila' ELSE 'Rest of PH' END"
AREAS  = ['Metro Manila', 'Rest of PH']

# min tests per province before it may enter a ranking. Set from the actual n distribution:
# fixed median n = 113k but the thin tail goes down to Apayao at 61 tests; mobile is far
# sparser (median 6.2k), so it needs a looser bar or the ranking has too few provinces left.
MIN_TESTS = {'broadband': 10_000, 'cellular': 5_000}

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.grid': True, 'grid.alpha': 0.25,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11,
})
C_FIXED, C_MOBILE = '#0B6E5C', '#B8571F'
C_AREA = {'Metro Manila': '#7B3FA0', 'Rest of PH': '#8C8C8C'}

split = q(f"""
    SELECT {AREA} AS area, COUNT(*) AS tests, COUNT(DISTINCT province) AS provinces
    FROM t WHERE province IS NOT NULL GROUP BY 1
""")
split['pct'] = (100 * split.tests / split.tests.sum()).round(1)
print(f'ROOT  : {ROOT}')
print(split.to_string(index=False))

# Part 1 · ช่องว่างเมืองหลวง–ต่างจังหวัด

ภาพหลักของเล่ม: Metro Manila เร็วกว่าส่วนที่เหลือของประเทศแค่ไหน

- **panel ซ้าย/กลาง:** ความเร็ว download แยก fixed vs mobile ทำทั้ง **mean** และ **median**
  (mean สูงกว่า median = การแจกแจงเบ้ขวา มี connection เร็ว ๆ ลากค่าเฉลี่ยขึ้น)
- **panel ขวา:** *mobile reliance* = สัดส่วน test ที่เป็น mobile — ยิ่งสูงยิ่งบ่งว่าพื้นที่นั้นพึ่งมือถือเพราะเน็ตบ้านเข้าไม่ถึง
- ตัวเลข **gap** ใต้กราฟคิดเป็น % ที่ Manila เร็วกว่า (`(manila − rest) / rest × 100`)

In [ ]:
gap = q(f"""
    SELECT {AREA} AS area,
           COUNT(*) AS tests,
           ROUND(100.0*SUM(CASE WHEN {MOBILE} THEN 1 ELSE 0 END)/COUNT(*), 1) AS mobile_reliance_pct,
           ROUND(AVG(CASE    WHEN {FIXED}  AND {DL} THEN mean_throughput_mbps END), 1) AS fixed_mean,
           ROUND(MEDIAN(CASE WHEN {FIXED}  AND {DL} THEN mean_throughput_mbps END), 1) AS fixed_med,
           ROUND(AVG(CASE    WHEN {MOBILE} AND {DL} THEN mean_throughput_mbps END), 1) AS mobile_mean,
           ROUND(MEDIAN(CASE WHEN {MOBILE} AND {DL} THEN mean_throughput_mbps END), 1) AS mobile_med
    FROM t
    WHERE province IS NOT NULL AND ({FIXED} OR {MOBILE})
    GROUP BY 1
""").set_index('area').loc[AREAS].reset_index()

x, w = np.arange(2), 0.38
fig, (axm, axmd, axr) = plt.subplots(1, 3, figsize=(16, 5.4),
                                     gridspec_kw={'width_ratios': [2, 2, 1]})

for ax, fcol, mcol, ttl in [(axm, 'fixed_mean', 'mobile_mean', 'Mean'),
                            (axmd, 'fixed_med', 'mobile_med', 'Median')]:
    b1 = ax.bar(x - w/2, gap[fcol], w, label='Fixed broadband', color=C_FIXED)
    b2 = ax.bar(x + w/2, gap[mcol], w, label='Mobile', color=C_MOBILE)
    ax.bar_label(b1, fmt='%.1f', padding=2)
    ax.bar_label(b2, fmt='%.1f', padding=2)
    ax.set_xticks(x)
    ax.set_xticklabels([f'{r.area}\n{r.tests/1e6:.0f}M tests' for _, r in gap.iterrows()])
    # % by which Manila beats the rest, per segment
    sub = [f'{c.split("_")[0]} {100*(gap[c][0]-gap[c][1])/gap[c][1]:+.0f}%' for c in (fcol, mcol)]
    ax.set_title(f'{ttl} download speed\n(Manila vs rest: {" · ".join(sub)})', fontsize=11)
axm.set_ylabel('Download (Mbps)')
axmd.sharey(axm)
axm.legend()

bb = axr.bar(x, gap['mobile_reliance_pct'], color=[C_AREA[a] for a in gap['area']])
axr.bar_label(bb, fmt='%.1f%%', padding=2)
axr.set_xticks(x)
axr.set_xticklabels(gap['area'])
axr.set_ylabel('% of tests that are mobile')
axr.set_title('Mobile reliance')

fig.suptitle('Metro Manila vs the rest of the Philippines (2023–2025)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

gap

# Part 2 · Top 10 / Bottom 10 จังหวัด

จัดอันดับแยกกันสองชุด: เรียงด้วย **median** และเรียงด้วย **mean** (คนละกราฟ ไม่ปนกัน)

> **ทำไมต้องมี `min_tests`:** ถ้าไม่กรอง bottom-10 จะกลายเป็นอันดับของ *จังหวัดที่ข้อมูลน้อย* ไม่ใช่ *จังหวัดที่เน็ตช้า* —
> ฝั่ง fixed มี Apayao ที่มีแค่ **61 test**, DinagatIslands 310, Sulu 696 ส่วนฝั่ง mobile บางกว่านั้นอีก
> (NuevaVizcaya 42, Palawan 49) ค่าพวกนี้เด้งได้ทั้งสองทาง จึงตั้งเกณฑ์ขั้นต่ำไว้ที่
> **fixed 10,000 · mobile 5,000 test** (ค่าใน `MIN_TESTS`) แล้วบอกจำนวนจังหวัดที่เหลือไว้ในหัวกราฟทุกครั้ง
- Metro Manila ระบายสีม่วงเพื่อให้เห็นว่าเมืองหลวงไปโผล่อันดับไหน · **n (จำนวน test) อยู่ที่ป้ายแกน y ข้างชื่อจังหวัด**

### Fixed broadband — เรียงด้วย median


In [ ]:
PROV_STATS = {}      # cache: one scan per network type, reused by both metrics

def province_stats(net_pred, net_key):
    """Per-province n / median / mean for one network type, filtered by MIN_TESTS."""
    if net_key not in PROV_STATS:
        d = q(f"""
            SELECT province,
                   COUNT(*)                     AS n,
                   MEDIAN(mean_throughput_mbps) AS med,
                   AVG(mean_throughput_mbps)    AS mean
            FROM t
            WHERE {net_pred} AND {DL} AND province IS NOT NULL
            GROUP BY 1
        """)
        PROV_STATS[net_key] = (d, d[d.n >= MIN_TESTS[net_key]].copy())
    return PROV_STATS[net_key]

def top_bottom_provinces(net_pred, net_key, seg_label, color, metric='med', topn=10):
    """Top-N and bottom-N provinces for one network type, ranked by `metric`
    ('med' or 'mean') — each metric gets its own figure so the two rankings
    can be compared side by side without being conflated in one chart."""
    d, kept = province_stats(net_pred, net_key)
    label = {'med': 'median', 'mean': 'mean'}[metric]
    ranked = kept.sort_values(metric).reset_index(drop=True)
    top, bot = ranked.tail(topn).iloc[::-1], ranked.head(topn)

    fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
    for ax, part, ttl in [(axes[0], top, f'Top {topn} — fastest'),
                          (axes[1], bot, f'Bottom {topn} — slowest')]:
        p = part.iloc[::-1].reset_index(drop=True)      # best-of-panel at the top
        y = np.arange(len(p))
        cols = [C_AREA['Metro Manila'] if s == MANILA else color for s in p['province']]
        ax.barh(y, p[metric], color=cols)
        mx = p[metric].max()
        for yi, r in zip(y, p.itertuples()):
            ax.text(getattr(r, metric) + mx*0.02, yi, f'{getattr(r, metric):.1f}',
                    va='center', fontsize=8)
        ax.set_yticks(y)
        ax.set_yticklabels([f'{r.province}\n(n={r.n:,})' for r in p.itertuples()], fontsize=7.5)
        ax.set_xlim(0, mx*1.18)
        ax.set_xlabel(f'{label.capitalize()} download (Mbps)')
        ax.set_title(ttl)
    fig.suptitle(f'{seg_label} — provincial ranking by {label.upper()}\n'
                 f'({len(kept)} of {len(d)} provinces pass min {MIN_TESTS[net_key]:,} tests · '
                 f'purple = Metro Manila)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    return ranked

fixed_by_median = top_bottom_provinces(FIXED, 'broadband', 'Fixed broadband', C_FIXED, metric='med')


### Fixed broadband — เรียงด้วย mean


In [ ]:
fixed_by_mean = top_bottom_provinces(FIXED, 'broadband', 'Fixed broadband', C_FIXED, metric='mean')


### Mobile — เรียงด้วย median

เกณฑ์ขั้นต่ำผ่อนเป็น 5,000 test เพราะข้อมูล mobile รายจังหวัดบางกว่า fixed มาก (median รายจังหวัด ~6,200 test เทียบกับ fixed ที่ ~113,000) — **จำนวนจังหวัดที่ผ่านเกณฑ์จึงน้อยกว่าฝั่ง fixed อย่างเห็นได้ชัด ให้อ่านหัวกราฟประกอบ**


In [ ]:
mobile_by_median = top_bottom_provinces(MOBILE, 'cellular', 'Mobile', C_MOBILE, metric='med')


### Mobile — เรียงด้วย mean


In [ ]:
mobile_by_mean = top_bottom_provinces(MOBILE, 'cellular', 'Mobile', C_MOBILE, metric='mean')


## PH-Specific Addition 4 — Paper Support Analysis

Ported in full, verbatim, from `ndt7_ph_paper.ipynb` (22 cells) — national + Manila/Cebu/Davao app-requirement threshold analysis (Table 5, Lübben & Misfeld 2022), speed and latency. Opens its own DuckDB connection, same as the source notebook.


## Part 0 · Setup

In [ ]:
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# --- resolve repo root regardless of the kernel's working dir ---
ROOT = Path.cwd()
while not (ROOT / 'data' / 'ph').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

CLEAN = ROOT / 'data' / 'ph' / 'mlab_ph_clean.parquet'
assert CLEAN.exists(), f'missing {CLEAN}'

con = duckdb.connect()
con.execute("SET max_memory='12GB'")
con.execute(f"SET temp_directory='{(ROOT / '.tmp').as_posix()}'")

# same monitoring/bot IP excluded in ndt7_ph_main.ipynb (865k tests, all SultanKudarat) --
# kept out here too so national numbers line up with the main EDA notebook
FLOOD_IPS = ('58.69.220.245',)
_flood = ', '.join(f"'{ip}'" for ip in FLOOD_IPS)
con.execute(f"CREATE OR REPLACE VIEW t AS SELECT * FROM read_parquet('{CLEAN.as_posix()}') "
            f"WHERE client_ip NOT IN ({_flood})")

def q(sql):
    return con.execute(sql).df()

FIXED  = "network_type = 'broadband'"
MOBILE = "network_type = 'cellular'"
DL     = "type = 'download'"

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.grid': True, 'grid.alpha': 0.25,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11,
})
C_FIXED, C_MOBILE = '#0B6E5C', '#B8571F'

n_rows = con.execute("SELECT COUNT(*) FROM t").fetchone()[0]
print(f'ROOT : {ROOT}')
print(f'CLEAN: {CLEAN.name} ({n_rows:,} rows, flood IP excluded)')

# Part 1 · แนวโน้มความเร็ว download ระดับประเทศ รายไตรมาส (2023 Q1 → 2025 Q4)

Time series ราย**ไตรมาส**ทั้ง 12 ไตรมาส แยก **fixed broadband vs mobile/cellular** — ทำทั้ง mean (ไวต่อ
connection เร็ว ๆ ที่เพิ่มเข้ามา) และ median (สะท้อนผู้ใช้ทั่วไป) เพื่อดูว่าเน็ตทั้งประเทศ "โต" จริงไหมและโตแบบไหน

- แกน x = ไตรมาส 2023Q1–2025Q4 (12 จุด) · แกน y = ความเร็ว download (Mbps)
- ตัวเลขบนจุดสุดท้ายของแต่ละเส้น = ค่า ณ 2025 Q4 (ใช้เทียบ endpoint growth ได้ตรงกับ Part 6 ของ `ndt7_ph_main.ipynb`)

In [ ]:
trend = q(f"""
    SELECT year,
           CAST(CEIL(month / 3.0) AS INT) AS quarter,
           CASE WHEN {FIXED} THEN 'Fixed' ELSE 'Mobile' END AS segment,
           COUNT(*)                              AS n,
           AVG(mean_throughput_mbps)              AS mean_dl,
           MEDIAN(mean_throughput_mbps)            AS median_dl
    FROM t
    WHERE {DL} AND ({FIXED} OR {MOBILE})
    GROUP BY 1, 2, 3
    ORDER BY 1, 2
""")
trend['qlabel'] = trend['year'].astype(str) + 'Q' + trend['quarter'].astype(str)
quarters = sorted(trend['qlabel'].unique(), key=lambda s: (int(s[:4]), int(s[5])))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 5.5), sharex=True)
for ax, col, ttl in [(a1, 'mean_dl', 'Mean download speed'), (a2, 'median_dl', 'Median download speed')]:
    for seg, color in [('Fixed', C_FIXED), ('Mobile', C_MOBILE)]:
        d = trend[trend.segment == seg].set_index('qlabel').reindex(quarters)
        ax.plot(quarters, d[col], marker='o', color=color, label=seg, lw=2)
        last = d[col].iloc[-1]
        ax.annotate(f'{last:.0f}', (len(quarters) - 1, last), textcoords='offset points',
                    xytext=(6, 0), va='center', color=color, fontsize=9, fontweight='bold')
    ax.set_title(ttl)
    ax.set_ylabel('Mbps')
    ax.set_xticks(range(len(quarters)))
    ax.set_xticklabels(quarters, rotation=45, ha='right', fontsize=8)
    ax.legend(loc='upper left')

fig.suptitle('Philippines — National Download Speed Trend by Quarter (2023 Q1 – 2025 Q4)',
             fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

trend.pivot(index='qlabel', columns='segment', values=['n', 'mean_dl', 'median_dl']).loc[quarters].round(1)

# Part 2 · แนวโน้ม/ระดับความเร็ว รายภูมิภาค (17 regions)

จาก Part 1 ที่ดูภาพรวมประเทศ ซูมลงมาราย **17 ภูมิภาค** เพื่อดูว่าตัวเลขระดับประเทศบัง variation ระหว่าง
ภูมิภาคไว้แค่ไหน — ใช้ `region` column ที่มากับ parquet + map ต่อไปยัง 3 หมู่เกาะ (Luzon/Visayas/Mindanao)
เดียวกับที่ `ndt7_ph_main.ipynb` Part 2–3 ใช้

metric: ค่าเฉลี่ยราย region *ต่อไตรมาส* แล้วเฉลี่ยข้าม 12 ไตรมาส (2023 Q1 – 2025 Q4) เพื่อไม่ให้ปี 2025
ที่มี test เยอะกลบปีอื่น

In [ ]:
from matplotlib.patches import Patch

# --- region -> island group (same mapping as ndt7_ph_main.ipynb) ---
ISLAND = {r: 'Luzon' for r in ['NCR', 'CAR', 'Region I', 'Region II', 'Region III',
                               'Region IV-A', 'MIMAROPA', 'Region V']}
ISLAND.update({r: 'Visayas'  for r in ['Region VI', 'Region VII', 'Region VIII']})
ISLAND.update({r: 'Mindanao' for r in ['Region IX', 'Region X', 'Region XI', 'Region XII',
                                       'Region XIII', 'BARMM']})

reg_map = q("""
    SELECT DISTINCT province, region
    FROM t
    WHERE province IS NOT NULL AND region IS NOT NULL
""")
_missing = sorted(set(reg_map['region']) - set(ISLAND))
assert not _missing, f'region(s) not in the ISLAND map: {_missing}'
reg_map['island'] = reg_map['region'].map(ISLAND)
con.register('reg_map', reg_map)

ISLAND_COLORS = {'Luzon': '#1f77b4', 'Visayas': '#ff7f0e', 'Mindanao': '#2ca02c'}
print(f'reg_map: {len(reg_map)} provinces -> {reg_map.region.nunique()} regions -> {reg_map.island.nunique()} islands')

In [ ]:
def region_speed_figure(net_pred, seg_label):
    """Region bar chart (mean | median panels) for one network type.
    metric = per-region test-weighted average per quarter, then averaged across all
    quarters 2023Q1-2025Q4; n (in bar) = total tests; dashed line = mean across regions."""
    agg = q(f"""
        WITH rq AS (
            SELECT m.region, m.island, t.year, CAST(CEIL(t.month/3.0) AS INT) AS quarter,
                   AVG(t.mean_throughput_mbps)    AS q_mean,
                   MEDIAN(t.mean_throughput_mbps) AS q_median,
                   COUNT(*)                       AS q_n
            FROM t JOIN reg_map m ON t.province = m.province
            WHERE {net_pred} AND {DL}
            GROUP BY 1, 2, 3, 4
        )
        SELECT region, island,
               AVG(q_mean)   AS mean_spd,
               AVG(q_median) AS median_spd,
               SUM(q_n)      AS total_tests
        FROM rq GROUP BY 1, 2
    """).sort_values('mean_spd').reset_index(drop=True)

    y = np.arange(len(agg))
    cols = [ISLAND_COLORS[i] for i in agg['island']]
    fig, axes = plt.subplots(1, 2, figsize=(15, 11), sharey=True, constrained_layout=True)
    for ax, mcol, ttl in [(axes[0], 'mean_spd', 'Mean'), (axes[1], 'median_spd', 'Median')]:
        ax.barh(y, agg[mcol], color=cols)
        ref, mx = agg[mcol].mean(), agg[mcol].max()
        ax.axvline(ref, ls='--', color='#444', lw=1)
        for yi, v, n in zip(y, agg[mcol], agg['total_tests']):
            ax.text(v + mx * 0.012, yi, f'{v:.0f}', va='center', fontsize=8)
            ax.text(mx * 0.012, yi, f'{n:,.0f}', va='center', fontsize=6.5, color='white')
        ax.set_title(f'{ttl}   (avg-of-region line = {ref:.0f})')
        ax.set_xlabel('Download Speed (Mbps)')
    axes[0].set_yticks(y)
    axes[0].set_yticklabels(agg['region'])
    axes[1].legend(handles=[Patch(color=c, label=l) for l, c in ISLAND_COLORS.items()],
                   loc='lower right', frameon=True)
    fig.suptitle(f'{seg_label} Download Speed by Region\n'
                 f'(weighted avg per quarter, averaged 2023 Q1 – 2025 Q4  ·  n = total tests, shown in bar)',
                 fontsize=13)
    plt.show()
    return agg

fixed_by_region = region_speed_figure(FIXED, 'Fixed Broadband')

# Part 3 · ISP มีทั้งหมดกี่เจ้า แยก broadband / cellular / hosting

นับ `isp` (ชื่อดิบตามที่ M-Lab ให้มา ไม่ merge แบรนด์) ที่ไม่ซ้ำกัน แยกตาม `network_type`

> **ข้อควรระวัง:** ผลรวมของ 3 หมวดจะ**มากกว่า**จำนวน ISP รวมทั้งหมด เพราะ ISP เดียวกันมีได้หลาย
> `network_type` (เช่น Globe/Converge รันทั้ง broadband, cellular และ hosting ในชื่อเดียว — ตรวจแล้วมี
> 81 ชื่อ ISP ที่คาบเกี่ยวมากกว่า 1 หมวด)

In [ ]:
total_isps = q("SELECT COUNT(DISTINCT isp) AS n FROM t WHERE isp IS NOT NULL")['n'][0]

by_type = q("""
    SELECT network_type,
           COUNT(DISTINCT isp) AS n_isps,
           COUNT(*)            AS tests
    FROM t
    WHERE isp IS NOT NULL
    GROUP BY 1
    ORDER BY tests DESC
""")

overlap = q("""
    SELECT COUNT(*) AS n
    FROM (
        SELECT isp
        FROM t
        WHERE isp IS NOT NULL
        GROUP BY 1
        HAVING COUNT(DISTINCT network_type) > 1
    )
""")['n'][0]

print(f'ISP ทั้งหมด (unique ชื่อ, ทุก network_type รวมกัน): {total_isps}')
print(f'ISP ที่คาบเกี่ยวมากกว่า 1 network_type: {overlap}\n')
by_type

# Part 4 · Top 10 ISP ต่อหมวด (broadband / cellular / hosting)

เหมือน Part 8 §1 ของ `ndt7_ph_main.ipynb` (top 10 ISP ตามจำนวน test) แต่เพิ่มหมวด **hosting** เข้ามาด้วย
เป็น 3 panel — แท่ง = จำนวน test (ล้าน) พร้อม % share ของหมวดนั้น

ตารางใต้กราฟ: rank, ISP, tests, share, unique client IP, tests/IP (เรียงตามอันดับ test เดียวกับกราฟ) — 
tests/IP สูงผิดปกติ = สัญญาณ IP เดียวยิง test ถี่ (ดู CGNAT note ใน main notebook Part 8)

In [ ]:
HOSTING = "network_type = 'hosting'"
C_HOSTING = '#6A4C93'

def isp_count_top(net_pred, topn=10):
    """Top-N ISPs by raw test volume for one network type, with unique client IPs
    alongside (tests != users: a few heavy retesters can inflate the test count)."""
    return q(f"""
        SELECT isp,
               COUNT(*)                  AS tests,
               COUNT(DISTINCT client_ip) AS uniq_ips
        FROM t
        WHERE {net_pred} AND isp IS NOT NULL
        GROUP BY 1
        ORDER BY tests DESC
        LIMIT {topn}
    """)

fig, axes = plt.subplots(1, 3, figsize=(19, 7))
isp_counts = {}
for ax, net_pred, c, ttl in [(axes[0], FIXED,   C_FIXED,   'Fixed broadband'),
                             (axes[1], MOBILE,  C_MOBILE,  'Mobile'),
                             (axes[2], HOSTING, C_HOSTING, 'Hosting')]:
    top = isp_count_top(net_pred)
    seg_total = q(f"SELECT COUNT(*) n FROM t WHERE {net_pred} AND isp IS NOT NULL")['n'][0]
    isp_counts[ttl] = top.assign(pct=100 * top['tests'] / seg_total,
                                 tests_per_ip=top['tests'] / top['uniq_ips'])

    d = top.iloc[::-1].reset_index(drop=True)          # biggest at the top of the bar chart
    y = np.arange(len(d))
    ax.barh(y, d['tests'] / 1e6, color=c)
    mx = (d['tests'] / 1e6).max()
    for yi, r in zip(y, d.itertuples()):
        ax.text(r.tests / 1e6 + mx * 0.02, yi,
                f'{r.tests/1e6:.1f}M ({100*r.tests/seg_total:.1f}%)', va='center', fontsize=7.5)
    ax.set_yticks(y)
    ax.set_yticklabels([s[:28] for s in d['isp']], fontsize=8)
    ax.set_xlabel('Tests (millions)')
    ax.set_xlim(0, mx * 1.35)
    ax.set_title(f'{ttl} — top {len(d)} by test count\n(total {seg_total/1e6:.1f}M tests)')

fig.suptitle('PH ISP size by test volume — fixed vs mobile vs hosting (2023–2025)',
             fontweight='bold', fontsize=13)
fig.tight_layout()
plt.show()

for seg, d in isp_counts.items():
    print(f'\n{seg} — top {len(d)} ISP')
    print(f'{"#":>2}  {"ISP":<44}{"tests":>13}{"share":>8}{"unique IPs":>13}{"tests/IP":>10}')
    print('-' * 92)
    for rank, r in enumerate(d.itertuples(), 1):
        print(f'{rank:>2}  {r.isp[:43]:<44}{r.tests:>13,}{r.pct:>7.1f}%'
              f'{r.uniq_ips:>13,}{r.tests_per_ip:>10.1f}')

# Part 5 · เน็ต PH พอสำหรับแอปพลิเคชันต่าง ๆ ไหม (Table 5, Lübben & Misfeld 2022)

อ้างอิง requirement จาก [Lübben & Misfeld 2022](docs/02-ndt7-mlab/lubben-misfeld-2022-mlab-german-internet-landscape.pdf)
Table 5 — Voice 64 kbps, Video HD 5 Mbps, Video UHD 25 Mbps, Cloud gaming 44 Mbps
(**ตัด latency ออกตามที่ขอ ดูแค่ download speed**)

**วิธีวัด — ตามแบบเดียวกับ paper ต้นทาง (§5.2):** ไม่ใช้แค่ mean/median เทียบ threshold ตัวเดียว (เพราะ
distribution เบ้ขวา ค่าเฉลี่ยประเทศไม่ได้แปลว่า "ผู้ใช้ทั่วไปทำได้") แต่ดู **% ของ test ที่ทำถึง requirement นั้นจริง**
— ตรงกับที่ paper เขียนว่า "roughly half of the measurements show a throughput of 25 Mbps...
much less achieve [44 Mbps]... more than 75% of tests" achieve the HD rate

ทำแยก fixed broadband กับ mobile เพราะเป็นคนละ market/technology

In [ ]:
REQUIREMENTS = [('Voice', 0.064), ('Video streaming (HD)', 5), ('Video streaming (UHD)', 25), ('Cloud gaming', 44)]

app_pass = q(f"""
    SELECT CASE WHEN {FIXED} THEN 'Fixed' ELSE 'Mobile' END AS segment,
           COUNT(*)                                AS n,
           AVG(mean_throughput_mbps)                AS mean_dl,
           MEDIAN(mean_throughput_mbps)              AS median_dl,
           100.0*SUM(CASE WHEN mean_throughput_mbps>=0.064 THEN 1 ELSE 0 END)/COUNT(*) AS pct_voice,
           100.0*SUM(CASE WHEN mean_throughput_mbps>=5     THEN 1 ELSE 0 END)/COUNT(*) AS pct_hd,
           100.0*SUM(CASE WHEN mean_throughput_mbps>=25    THEN 1 ELSE 0 END)/COUNT(*) AS pct_uhd,
           100.0*SUM(CASE WHEN mean_throughput_mbps>=44    THEN 1 ELSE 0 END)/COUNT(*) AS pct_gaming
    FROM t
    WHERE {DL} AND ({FIXED} OR {MOBILE})
    GROUP BY 1
""").set_index('segment')

pct_cols = ['pct_voice', 'pct_hd', 'pct_uhd', 'pct_gaming']
fig, ax = plt.subplots(figsize=(10, 5.5))
x = np.arange(len(REQUIREMENTS))
w = 0.35
for i, (seg, color) in enumerate([('Fixed', C_FIXED), ('Mobile', C_MOBILE)]):
    vals = app_pass.loc[seg, pct_cols].values.astype(float)
    bars = ax.bar(x + (i - 0.5) * w, vals, width=w, color=color, label=seg)
    for xi, v in zip(x + (i - 0.5) * w, vals):
        ax.text(xi, v + 1.5, f'{v:.0f}%', ha='center', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels([f'{name}\n(≥{req} Mbps)' for name, req in REQUIREMENTS])
ax.set_ylabel('% of download tests meeting the requirement')
ax.set_ylim(0, 108)
ax.axhline(50, ls='--', color='#888', lw=1)
ax.legend()
ax.set_title('Philippines — Share of Tests Meeting Application Data-Rate Requirements\n'
             '(Table 5, Lübben & Misfeld 2022 — latency not evaluated)', fontsize=12)
fig.tight_layout()
plt.show()

app_pass[['n', 'mean_dl', 'median_dl'] + pct_cols].round(1)

# Part 6 · เพิ่ม latency (RTT) เทียบ requirement เดียวกัน (แยกกราฟจาก Part 5)

Table 5 มี latency requirement แค่ 2 ค่าที่เป็นตัวเลขจริง — **Voice ≤200 ms** และ **Cloud gaming ≤25 ms**
(Video streaming ระบุแค่ "few seconds" ซึ่งไม่ใช่ threshold ที่วัด pass/fail ได้ เลยไม่รวมในกราฟนี้)

- ใช้ `min_rtt` จาก NDT7 (RTT วัดจาก TCP stack, round-trip ไป-กลับ) — **paper ต้นทางประมาณ one-way latency
  เป็นครึ่งหนึ่งของ RTT** ("latency, approximated with half of the RTT") จึงใช้สูตรเดียวกันตรงนี้:
  `latency ≈ min_rtt / 2`
- **ข้อควรระวัง:** นี่เป็นการประมาณคร่าว ๆ (สมมติ path สมมาตรทั้งไปและกลับ) ถ้าใช้ RTT เต็มไม่หารสอง
  % ผ่าน cloud-gaming tier จะลดลงเกือบครึ่ง (เช่น fixed 48.8% → 24.3%) — ตัวเลขในกราฟนี้ใช้สูตร**หารสอง**
  ตามที่ paper ต้นทางทำ เพื่อให้เทียบกันได้ตรง ๆ

In [ ]:
LAT_REQUIREMENTS = [('Voice', 200), ('Cloud gaming', 25)]

rtt_pass = q("""
    SELECT CASE WHEN network_type='broadband' THEN 'Fixed' ELSE 'Mobile' END AS segment,
           COUNT(*)                                                    AS n,
           AVG(CASE WHEN min_rtt < 2000 THEN min_rtt END)               AS mean_rtt,
           MEDIAN(min_rtt)                                              AS median_rtt,
           100.0*SUM(CASE WHEN min_rtt/2.0 <= 200 THEN 1 ELSE 0 END)/COUNT(*) AS pct_voice_200,
           100.0*SUM(CASE WHEN min_rtt/2.0 <= 25  THEN 1 ELSE 0 END)/COUNT(*) AS pct_gaming_25
    FROM t
    WHERE network_type IN ('broadband', 'cellular') AND min_rtt IS NOT NULL AND min_rtt > 0
    GROUP BY 1
""").set_index('segment')

lat_cols = ['pct_voice_200', 'pct_gaming_25']
fig, ax = plt.subplots(figsize=(7, 5.5))
x = np.arange(len(LAT_REQUIREMENTS))
w = 0.35
for i, (seg, color) in enumerate([('Fixed', C_FIXED), ('Mobile', C_MOBILE)]):
    vals = rtt_pass.loc[seg, lat_cols].values.astype(float)
    ax.bar(x + (i - 0.5) * w, vals, width=w, color=color, label=seg)
    for xi, v in zip(x + (i - 0.5) * w, vals):
        ax.text(xi, v + 1.5, f'{v:.0f}%', ha='center', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels([f'{name}\n(≤{req} ms)' for name, req in LAT_REQUIREMENTS])
ax.set_ylabel('% of tests meeting the latency requirement')
ax.set_ylim(0, 108)
ax.axhline(50, ls='--', color='#888', lw=1)
ax.legend()
ax.set_title('Philippines — Share of Tests Meeting Application Latency Requirements\n'
             '(latency ≈ min_rtt / 2, per Lübben & Misfeld 2022 methodology)', fontsize=12)
fig.tight_layout()
plt.show()

rtt_pass[['n', 'mean_rtt', 'median_rtt'] + lat_cols].round(1)

# Part 7 · Manila – Cebu – Davao (3 จังหวัดใหญ่สุดของ PH)

3 province (ADM1) ที่มี test volume มากสุด — Metropolitan Manila, Cebu, Davao del Sur — บังเอิญเป็น
province ตัวแทนคนละหมู่เกาะพอดี (Luzon/Visayas/Mindanao) และครอบคลุมเมืองเด่น Quezon City/Manila/Makati,
Cebu City, Davao City ตามลำดับ

metric เดียวกับ Part 2 (region): ค่าเฉลี่ยราย province *ต่อไตรมาส* แล้วเฉลี่ยข้าม 12 ไตรมาส (ไม่ให้ปี 2025
ที่ test เยอะกลบปีอื่น) — แยก **fixed broadband vs mobile**, ทำทั้ง mean และ median

In [ ]:
BIG3 = {'MetropolitanManila': 'Manila', 'Cebu': 'Cebu', 'DavaodelSur': 'Davao'}
big3_order = ['Manila', 'Cebu', 'Davao']

big3 = q(f"""
    WITH pq AS (
        SELECT province,
               CASE WHEN {FIXED} THEN 'Fixed' ELSE 'Mobile' END AS segment,
               year, CAST(CEIL(month / 3.0) AS INT) AS quarter,
               AVG(mean_throughput_mbps)    AS q_mean,
               MEDIAN(mean_throughput_mbps) AS q_median,
               COUNT(*)                     AS q_n
        FROM t
        WHERE province IN ('MetropolitanManila', 'Cebu', 'DavaodelSur')
          AND {DL} AND ({FIXED} OR {MOBILE})
        GROUP BY 1, 2, 3, 4
    )
    SELECT province, segment,
           AVG(q_mean)   AS mean_spd,
           AVG(q_median) AS median_spd,
           SUM(q_n)      AS total_tests
    FROM pq GROUP BY 1, 2
""")
big3['prov'] = big3['province'].map(BIG3)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 5.5), sharey=False)
x = np.arange(len(big3_order))
w = 0.35
for ax, col, ttl in [(a1, 'mean_spd', 'Mean download speed'), (a2, 'median_spd', 'Median download speed')]:
    for i, (seg, color) in enumerate([('Fixed', C_FIXED), ('Mobile', C_MOBILE)]):
        d = big3[big3.segment == seg].set_index('prov').reindex(big3_order)
        ax.bar(x + (i - 0.5) * w, d[col], width=w, color=color, label=seg)
        for xi, v in zip(x + (i - 0.5) * w, d[col]):
            ax.text(xi, v + d[col].max() * 0.015, f'{v:.0f}', ha='center', fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(big3_order)
    ax.set_ylabel('Mbps')
    ax.set_title(ttl)
    ax.legend()

fig.suptitle('Manila – Cebu – Davao — Download Speed by Province\n'
             '(weighted avg per quarter, averaged 2023 Q1 – 2025 Q4)',
             fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

big3.pivot(index='prov', columns='segment', values=['total_tests', 'mean_spd', 'median_spd']).loc[big3_order].round(1)

# Part 8 · Manila – Cebu – Davao ตอบโจทย์แอปพลิเคชันไหน (speed, Table 5)

เหมือน Part 5 (ระดับประเทศ) แต่ตัดเฉพาะ 3 จังหวัดใหญ่ที่เลือกไว้ — ดู % ของ test ที่ทำความเร็วถึง
requirement แต่ละ tier แยกราย province (สีเดียวกับ Part 2: Manila=Luzon, Cebu=Visayas, Davao=Mindanao)
ทำแยก panel Fixed / Mobile เพราะเป็นคนละตลาด

In [ ]:
BIG3_COLORS = {'Manila': ISLAND_COLORS['Luzon'], 'Cebu': ISLAND_COLORS['Visayas'], 'Davao': ISLAND_COLORS['Mindanao']}

prov_app_pass = q(f"""
    SELECT province,
           CASE WHEN {FIXED} THEN 'Fixed' ELSE 'Mobile' END AS segment,
           COUNT(*)                                AS n,
           100.0*SUM(CASE WHEN mean_throughput_mbps>=0.064 THEN 1 ELSE 0 END)/COUNT(*) AS pct_voice,
           100.0*SUM(CASE WHEN mean_throughput_mbps>=5     THEN 1 ELSE 0 END)/COUNT(*) AS pct_hd,
           100.0*SUM(CASE WHEN mean_throughput_mbps>=25    THEN 1 ELSE 0 END)/COUNT(*) AS pct_uhd,
           100.0*SUM(CASE WHEN mean_throughput_mbps>=44    THEN 1 ELSE 0 END)/COUNT(*) AS pct_gaming
    FROM t
    WHERE province IN ('MetropolitanManila', 'Cebu', 'DavaodelSur')
      AND {DL} AND ({FIXED} OR {MOBILE})
    GROUP BY 1, 2
""")
prov_app_pass['prov'] = prov_app_pass['province'].map(BIG3)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 5.5), sharey=True)
x = np.arange(len(REQUIREMENTS))
w = 0.26
for ax, seg in [(a1, 'Fixed'), (a2, 'Mobile')]:
    d = prov_app_pass[prov_app_pass.segment == seg].set_index('prov').reindex(big3_order)
    for i, prov in enumerate(big3_order):
        vals = d.loc[prov, pct_cols].values.astype(float)
        ax.bar(x + (i - 1) * w, vals, width=w, color=BIG3_COLORS[prov], label=prov)
        for xi, v in zip(x + (i - 1) * w, vals):
            ax.text(xi, v + 1.5, f'{v:.0f}', ha='center', fontsize=7.5)
    ax.set_xticks(x)
    ax.set_xticklabels([f'{name}\n(≥{req} Mbps)' for name, req in REQUIREMENTS])
    ax.set_title(seg)
    ax.axhline(50, ls='--', color='#888', lw=1)
    ax.legend()
a1.set_ylabel('% of download tests meeting the requirement')
a1.set_ylim(0, 108)

fig.suptitle('Manila – Cebu – Davao — Share of Tests Meeting Application Data-Rate Requirements\n'
             '(Table 5, Lübben & Misfeld 2022)', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

prov_app_pass.pivot(index='prov', columns='segment', values=['n'] + pct_cols).loc[big3_order].round(1)

# Part 9 · Manila – Cebu – Davao ตอบโจทย์แอปพลิเคชันไหน (latency, Table 5)

เหมือน Part 6 (ระดับประเทศ) แต่ตัดเฉพาะ 3 จังหวัด — latency ≈ `min_rtt / 2` เทียบ Voice (≤200ms) และ
Cloud gaming (≤25ms) เท่านั้น (Video streaming ไม่มี threshold ตัวเลขจริง)

In [ ]:
prov_rtt_pass = q("""
    SELECT province,
           CASE WHEN network_type='broadband' THEN 'Fixed' ELSE 'Mobile' END AS segment,
           COUNT(*)                                                    AS n,
           100.0*SUM(CASE WHEN min_rtt/2.0 <= 200 THEN 1 ELSE 0 END)/COUNT(*) AS pct_voice_200,
           100.0*SUM(CASE WHEN min_rtt/2.0 <= 25  THEN 1 ELSE 0 END)/COUNT(*) AS pct_gaming_25
    FROM t
    WHERE province IN ('MetropolitanManila', 'Cebu', 'DavaodelSur')
      AND network_type IN ('broadband', 'cellular') AND min_rtt IS NOT NULL AND min_rtt > 0
    GROUP BY 1, 2
""")
prov_rtt_pass['prov'] = prov_rtt_pass['province'].map(BIG3)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 5.5), sharey=True)
x = np.arange(len(LAT_REQUIREMENTS))
w = 0.26
for ax, seg in [(a1, 'Fixed'), (a2, 'Mobile')]:
    d = prov_rtt_pass[prov_rtt_pass.segment == seg].set_index('prov').reindex(big3_order)
    for i, prov in enumerate(big3_order):
        vals = d.loc[prov, lat_cols].values.astype(float)
        ax.bar(x + (i - 1) * w, vals, width=w, color=BIG3_COLORS[prov], label=prov)
        for xi, v in zip(x + (i - 1) * w, vals):
            ax.text(xi, v + 1.5, f'{v:.0f}', ha='center', fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels([f'{name}\n(≤{req} ms)' for name, req in LAT_REQUIREMENTS])
    ax.set_title(seg)
    ax.axhline(50, ls='--', color='#888', lw=1)
    ax.legend()
a1.set_ylabel('% of tests meeting the latency requirement')
a1.set_ylim(0, 108)

fig.suptitle('Manila – Cebu – Davao — Share of Tests Meeting Application Latency Requirements\n'
             '(latency ≈ min_rtt / 2)', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

prov_rtt_pass.pivot(index='prov', columns='segment', values=['n'] + lat_cols).loc[big3_order].round(1)